In [5]:
import torch, os

print("CUDA:", torch.cuda.is_available())
print("Working files:", os.listdir("/kaggle/working")[:20])

CUDA: True
Working files: ['.virtual_documents']


In [4]:
import glob
import os

model_files = []

for ext in ["*.pth", "*.pt", "*.ckpt", "*.bin", "*.safetensors"]:
    model_files.extend(
        glob.glob(f"/kaggle/input/**/*{ext[1:]}", recursive=True)
    )

for p in model_files:
    print(p)

/kaggle/input/models/ademizhan/cerebro-foundation-best/pytorch/default/1/cerebro_foundation_best.pth


In [1]:
!pip uninstall -y torch torchvision torchaudio

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 93.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 76.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 177.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 96.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 212.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 187.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 140.6

In [6]:
import torch, platform, psutil

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("RAM:", round(psutil.virtual_memory().total / 1024**3, 2), "GB")

Python: 3.12.12
PyTorch: 2.10.0+cu128
PyTorch CUDA: 12.8
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
RAM: 31.35 GB


In [7]:
import os
import gc
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.signal import find_peaks

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

Device: cuda
CUDA available: True


In [8]:
import gdown
import numpy as np

# For intra-subject

def download_dataset(file_id):
    download_url = f"https://drive.google.com/uc?id={file_id}"
    output_file = "mimic.npy"
    gdown.download(download_url, output_file, quiet=False)
    data = np.load("mimic.npy", allow_pickle=True).item()
    return data

def load_dataset(dataset):
    X = []  # ECG segments
    y = []  # ABP segments
# Loop over each patient
    for patient_id, signals in dataset.items():
        ecg_segments = signals['ecg']
        abp_segments = signals['abp']
        # Safety check: skip if segment lengths don't match
        if len(ecg_segments) != len(abp_segments):
            continue

        for ecg_seg, abp_seg in zip(ecg_segments, abp_segments):
            if (
                isinstance(ecg_seg, np.ndarray) and ecg_seg.shape == (625,) and
                isinstance(abp_seg, np.ndarray) and abp_seg.shape == (625,)
            ):
                X.append(ecg_seg)
                y.append(abp_seg)


    # Convert lists to numpy arrays
    X = np.array(X)
    y = np.array(y)
    
    return X, y


links = {
    # this is all from mimic-iii dataset
    "no_pp": "18nRE4Z_RBKjT3grYFUiCV0cjUw6gZNd5",
    "bw": "1mui4ifqbZv9e-5GeE4XyU061Kxi-wZdN",
    "bw+maf": "1wRU1UTPOzVqa_2lIm8GLI94jeLdf4pgh",
    "dwt": "1HYGwjSBrgiMV-4CSOiMxJXFyNzlSDBq8",
    "nk2": "1ZrELLKXvmhsuPhirAcvsZ4TcXtRC1rGQ",

    # this is from mimic-iv
    "no_pp_m4":"1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY",
    "bw_m4":"1tK1fPxBhZicWW5Vd5S8DFrKefnHQFVjQ",
    "bw+maf_m4":"12KmtjeVHuZP4omk-AvEBpfwmI1cPn1R7",
    "nk2_m4":"12AHW7_3ytaBs34hGHsIJ-ruRC8vpNE9q",
    "dwt_m4":"1Mmb9NxnwvKBSa1dmNJtAcF63V8N8BxYR"
  }
data = download_dataset(links["no_pp_m4"])
X,y = load_dataset(data)
print(X.shape, y.shape)

Downloading...
From (original): https://drive.google.com/uc?id=1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY
From (redirected): https://drive.google.com/uc?id=1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY&confirm=t&uuid=0d3439c7-fa57-4ecc-8ca4-31caf34950cb
To: /kaggle/working/mimic.npy
100%|██████████| 384M/384M [00:07<00:00, 53.1MB/s] 


(25319, 625) (25319, 625)


In [14]:
import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"

cerebro_pretrained_path = "/kaggle/input/models/ademizhan/cerebro-foundation-best/pytorch/default/1/cerebro_foundation_best.pth"

model_cerebro_full = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
).to(device)

pretrained_dict = torch.load(
    cerebro_pretrained_path,
    map_location="cpu"
)

if isinstance(pretrained_dict, dict) and "model_state_dict" in pretrained_dict:
    pretrained_dict = pretrained_dict["model_state_dict"]

model_dict = model_cerebro_full.state_dict()
matched_state_cerebro = {}

for k, v in pretrained_dict.items():
    candidates = [
        k,
        k.replace("module.", ""),
        "model." + k,
        "foundation." + k
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == v.shape:
            matched_state_cerebro[candidate] = v
            break

missing, unexpected = model_cerebro_full.load_state_dict(
    matched_state_cerebro,
    strict=False
)

model_cerebro_full = model_cerebro_full.to(device)

print("CUDA:", torch.cuda.is_available())
print("Device:", device)
print("Loaded tensors:", len(matched_state_cerebro))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

total_params_cerebro = sum(p.numel() for p in model_cerebro_full.parameters())
trainable_params_cerebro = sum(p.numel() for p in model_cerebro_full.parameters() if p.requires_grad)

print("Total parameters:", total_params_cerebro)
print("Trainable parameters:", trainable_params_cerebro)

CUDA: True
Device: cuda
Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0
Total parameters: 6637361
Trainable parameters: 6637361


In [11]:
class AlternatingAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)

    def forward(self, x, B, C, Np):
        x = x.view(B * C, Np, -1)
        x, _ = self.attn(x, x, x)

        x = x.view(B, C, Np, -1).transpose(1, 2).reshape(B * Np, C, -1)
        x, _ = self.attn(x, x, x)

        x = x.view(B, Np, C, -1).transpose(1, 2).reshape(B, C * Np, -1)
        return x


class CEReBrO_Foundation(nn.Module):
    def __init__(self, num_channels=2, patch_size=64, dim=192, depth=8, heads=12, mlp_dim=768, mode="pretrain"):
        super().__init__()
        self.patch_size = patch_size
        self.dim = dim
        self.Np = 625 // patch_size
        self.mode = mode

        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.patch_embed = nn.Linear(patch_size, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_channels * self.Np, dim))

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "attn": AlternatingAttention(dim, heads),
                "ffn": nn.Sequential(
                    nn.Linear(dim, mlp_dim),
                    nn.GELU(),
                    nn.Linear(mlp_dim, dim)
                ),
                "norm1": nn.LayerNorm(dim),
                "norm2": nn.LayerNorm(dim)
            }) for _ in range(depth)
        ])

        self.recon_head = nn.Linear(dim, patch_size)
        self.regression_head = nn.Linear(num_channels * self.Np * dim, 625)

    def forward(self, x, mask_binary=None, mode=None):
        B, C, L = x.shape
        x_patches = x[:, :, :self.Np * self.patch_size].view(B, C, self.Np, self.patch_size)
        x_enc = self.patch_embed(x_patches)

        current_mode = mode if mode is not None else self.mode

        if current_mode == "pretrain" and mask_binary is not None:
            m_tokens = self.mask_token.expand(B, C, self.Np, -1)
            mask_reshaped = mask_binary.view(B, 1, self.Np, 1)
            x_enc = x_enc * mask_reshaped + m_tokens * (1 - mask_reshaped)

        x_enc = x_enc.view(B, C * self.Np, self.dim) + self.pos_embed

        for layer in self.layers:
            x_enc = x_enc + layer["attn"](layer["norm1"](x_enc), B, C, self.Np)
            x_enc = x_enc + layer["ffn"](layer["norm2"](x_enc))

        if current_mode == "regression":
            x_flat = x_enc.reshape(B, -1)
            return self.regression_head(x_flat)
        else:
            return self.recon_head(x_enc.view(B, C, self.Np, self.dim)).view(B, C, -1)

In [12]:
def load_dataset_multi(dataset):
    X, y = [], []

    for patient_id, signals in dataset.items():
        ecg_segs = signals.get("ecg", [])
        ppg_segs = signals.get("ppg", [])
        abp_segs = signals.get("abp", [])

        for e, p, a in zip(ecg_segs, ppg_segs, abp_segs):
            if all(isinstance(s, np.ndarray) and s.shape == (625,) for s in [e, p, a]):
                X.append(np.stack([e, p], axis=0))
                y.append(a)

    return np.array(X), np.array(y)


dataset = np.load("mimic.npy", allow_pickle=True).item()
X, y = load_dataset_multi(dataset)

print("Loaded segments:", len(X))
print("X shape:", X.shape)
print("y shape:", y.shape)

Loaded segments: 25319
X shape: (25319, 2, 625)
y shape: (25319, 625)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=seed
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=seed
)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
X_val = x_scaler.transform(X_val.reshape(-1, 1)).reshape(X_val.shape)
X_test = x_scaler.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)

y_train = y_scaler.fit_transform(y_train.reshape(-1, 1)).reshape(y_train.shape)
y_val = y_scaler.transform(y_val.reshape(-1, 1)).reshape(y_val.shape)
y_test = y_scaler.transform(y_test.reshape(-1, 1)).reshape(y_test.shape)

train_loader = DataLoader(
    TensorDataset(torch.Tensor(X_train), torch.Tensor(y_train)),
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(torch.Tensor(X_val), torch.Tensor(y_val)),
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(torch.Tensor(X_test), torch.Tensor(y_test)),
    batch_size=64,
    shuffle=False
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (16204, 2, 625) (16204, 625)
Val: (4051, 2, 625) (4051, 625)
Test: (5064, 2, 625) (5064, 625)


In [16]:
class WaveformSBPDBPLoss(nn.Module):
    def __init__(self, waveform_weight=1.0, sbp_weight=0.5, dbp_weight=0.5):
        super().__init__()
        self.waveform_weight = waveform_weight
        self.sbp_weight = sbp_weight
        self.dbp_weight = dbp_weight
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        if pred.shape != target.shape:
            pred = pred.view_as(target)

        waveform_loss = self.mse(pred, target)

        pred_sbp = torch.max(pred, dim=-1).values
        target_sbp = torch.max(target, dim=-1).values

        pred_dbp = torch.min(pred, dim=-1).values
        target_dbp = torch.min(target, dim=-1).values

        sbp_loss = self.mse(pred_sbp, target_sbp)
        dbp_loss = self.mse(pred_dbp, target_dbp)

        return (
            self.waveform_weight * waveform_loss
            + self.sbp_weight * sbp_loss
            + self.dbp_weight * dbp_loss
        )

In [17]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for bx, by in loader:
        bx = bx.to(device).float()
        by = by.to(device).float()

        optimizer.zero_grad()
        pred = model(bx)

        if pred.shape != by.shape:
            pred = pred.view_as(by)

        loss = criterion(pred, by)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )

        optimizer.step()
        total_loss += loss.item() * bx.size(0)

    return total_loss / len(loader.dataset)


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device).float()
            by = by.to(device).float()

            pred = model(bx)

            if pred.shape != by.shape:
                pred = pred.view_as(by)

            loss = criterion(pred, by)
            total_loss += loss.item() * bx.size(0)

    return total_loss / len(loader.dataset)

In [19]:
import os
import random
import numpy as np
import torch

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print(e)

print("Seed fixed:", SEED)

Seed fixed: 42


In [20]:
for param in model_cerebro_full.parameters():
    param.requires_grad = True

criterion_cerebro_full = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_cerebro_full = torch.optim.AdamW(
    model_cerebro_full.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

num_epochs_cerebro_full = 30
best_val_loss_cerebro_full = float("inf")

best_model_path_cerebro_full = "cerebro_full_ft_sbpdbp_loss_seed42.pth"
history_cerebro_full = []

for epoch in range(num_epochs_cerebro_full):
    train_loss = train_one_epoch(
        model_cerebro_full,
        train_loader,
        optimizer_cerebro_full,
        criterion_cerebro_full,
        device
    )

    val_loss = validate_one_epoch(
        model_cerebro_full,
        val_loader,
        criterion_cerebro_full,
        device
    )

    history_cerebro_full.append({
        "method": "Cerebro_full_finetuning_sbpdbp_loss",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 1e-4,
        "weight_decay": 1e-2,
        "seed": SEED
    })

    if val_loss < best_val_loss_cerebro_full:
        best_val_loss_cerebro_full = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": "Cerebro_full_finetuning_sbpdbp_loss",
            "model_state_dict": model_cerebro_full.state_dict(),
            "optimizer_state_dict": optimizer_cerebro_full.state_dict(),
            "best_val_loss": float(best_val_loss_cerebro_full),
            "lr": 1e-4,
            "weight_decay": 1e-2,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_cerebro_full)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_cerebro_full}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Epoch [1/30] | Train Loss: 0.908289 | Val Loss: 0.923751 saved
Epoch [2/30] | Train Loss: 0.781916 | Val Loss: 0.772427 saved
Epoch [3/30] | Train Loss: 0.670090 | Val Loss: 0.722092 saved
Epoch [4/30] | Train Loss: 0.588331 | Val Loss: 0.644346 saved
Epoch [5/30] | Train Loss: 0.517220 | Val Loss: 0.576073 saved
Epoch [6/30] | Train Loss: 0.463693 | Val Loss: 0.543680 saved
Epoch [7/30] | Train Loss: 0.416170 | Val Loss: 0.516765 saved
Epoch [8/30] | Train Loss: 0.382446 | Val Loss: 0.498916 saved
Epoch [9/30] | Train Loss: 0.354825 | Val Loss: 0.451898 saved
Epoch [10/30] | Train Loss: 0.328559 | Val Loss: 0.460891 
Epoch [11/30] | Train Loss: 0.308332 | Val Loss: 0.435126 saved
Epoch [12/30] | Train Loss: 0.290318 | Val Loss: 0.422690 saved
Epoch [13/30] | Train Loss: 0.273799 | Val Loss: 0.426785 
Epoch [14/30] | Train Loss: 0.259352 | Val Loss: 0.392986 saved
Epoch [15/30] | Train Loss: 0.248695 | Val Loss: 0.383876 saved
Epoch [16/30] | Train Loss: 0.234369 | Val Loss: 0.379623 s

In [21]:
checkpoint_cerebro_full = torch.load(
    best_model_path_cerebro_full,
    map_location="cpu"
)

model_cerebro_full_best = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
)

model_cerebro_full_best.load_state_dict(
    checkpoint_cerebro_full["model_state_dict"]
)

model_cerebro_full_best = model_cerebro_full_best.to(device)
model_cerebro_full_best.eval()

print("Loaded best Cerebro full fine-tuned model")
print("Best epoch:", checkpoint_cerebro_full["epoch"])
print("Best val loss:", checkpoint_cerebro_full["best_val_loss"])

Loaded best Cerebro full fine-tuned model
Best epoch: 30
Best val loss: 0.30469589359125837


In [22]:
def evaluate_model_waveform_cerebro(model, loader, y_scaler, device):
    model.eval()

    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device).float()

            pred = model(xb)

            pred = pred.detach().cpu().numpy()
            true = yb.detach().cpu().numpy()

            pred = pred.reshape(pred.shape[0], -1)
            true = true.reshape(true.shape[0], -1)

            y_pred_all.append(pred)
            y_true_all.append(true)

    y_pred_all = np.concatenate(y_pred_all, axis=0)
    y_true_all = np.concatenate(y_true_all, axis=0)

    y_pred_real = y_scaler.inverse_transform(
        y_pred_all.reshape(-1, 1)
    ).reshape(y_pred_all.shape)

    y_true_real = y_scaler.inverse_transform(
        y_true_all.reshape(-1, 1)
    ).reshape(y_true_all.shape)

    return y_true_real, y_pred_real

In [23]:
y_true_cerebro_full, y_pred_cerebro_full = evaluate_model_waveform_cerebro(
    model_cerebro_full_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_cerebro_full.shape)
print("y_pred:", y_pred_cerebro_full.shape)

print("true min/max:", y_true_cerebro_full.min(), y_true_cerebro_full.max())
print("pred min/max:", y_pred_cerebro_full.min(), y_pred_cerebro_full.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 16.239668 188.42444


In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def extract_sbp_dbp(signal):
    signal = np.asarray(signal).squeeze()
    sbp = np.max(signal)
    dbp = np.min(signal)
    return sbp, dbp


sbp_true_cerebro_full = []
dbp_true_cerebro_full = []
sbp_pred_cerebro_full = []
dbp_pred_cerebro_full = []

for true_sig, pred_sig in zip(y_true_cerebro_full, y_pred_cerebro_full):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_cerebro_full.append(sbp_t)
    dbp_true_cerebro_full.append(dbp_t)
    sbp_pred_cerebro_full.append(sbp_p)
    dbp_pred_cerebro_full.append(dbp_p)

sbp_true_cerebro_full = np.array(sbp_true_cerebro_full)
dbp_true_cerebro_full = np.array(dbp_true_cerebro_full)
sbp_pred_cerebro_full = np.array(sbp_pred_cerebro_full)
dbp_pred_cerebro_full = np.array(dbp_pred_cerebro_full)

sbp_mae_cerebro_full = mean_absolute_error(
    sbp_true_cerebro_full,
    sbp_pred_cerebro_full
)

dbp_mae_cerebro_full = mean_absolute_error(
    dbp_true_cerebro_full,
    dbp_pred_cerebro_full
)

sbp_rmse_cerebro_full = np.sqrt(mean_squared_error(
    sbp_true_cerebro_full,
    sbp_pred_cerebro_full
))

dbp_rmse_cerebro_full = np.sqrt(mean_squared_error(
    dbp_true_cerebro_full,
    dbp_pred_cerebro_full
))

sbp_r2_cerebro_full = r2_score(
    sbp_true_cerebro_full,
    sbp_pred_cerebro_full
)

dbp_r2_cerebro_full = r2_score(
    dbp_true_cerebro_full,
    dbp_pred_cerebro_full
)

avg_mae_cerebro_full = (
    sbp_mae_cerebro_full + dbp_mae_cerebro_full
) / 2

print("Cerebro Full Fine-Tuning with SBP/DBP Loss Results")
print("==================================================")
print("Setting: Direct full fine-tuning, LR 1e-4")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_cerebro_full:.3f}")
print(f"RMSE : {sbp_rmse_cerebro_full:.3f}")
print(f"R2   : {sbp_r2_cerebro_full:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_cerebro_full:.3f}")
print(f"RMSE : {dbp_rmse_cerebro_full:.3f}")
print(f"R2   : {dbp_r2_cerebro_full:.4f}")

print("\nTable format:")
print(f"{sbp_mae_cerebro_full:.2f}/{dbp_mae_cerebro_full:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_cerebro_full:.3f}")

Cerebro Full Fine-Tuning with SBP/DBP Loss Results
Setting: Direct full fine-tuning, LR 1e-4
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 5.999
RMSE : 8.486
R2   : 0.6745

DBP
MAE  : 3.112
RMSE : 4.784
R2   : 0.5412

Table format:
6.00/3.11

Average MAE:
4.556


In [25]:
results_cerebro_full = pd.DataFrame([{
    "model": "Cerebro",
    "checkpoint": "cerebro_foundation_best",
    "fine_tuning": "full fine-tuning",
    "loss": "Waveform + SBP + DBP loss",
    "seed": SEED,
    "best_epoch": checkpoint_cerebro_full["epoch"],
    "best_val_loss": checkpoint_cerebro_full["best_val_loss"],
    "lr": checkpoint_cerebro_full["lr"],
    "weight_decay": checkpoint_cerebro_full["weight_decay"],
    "sbp_mae": sbp_mae_cerebro_full,
    "dbp_mae": dbp_mae_cerebro_full,
    "avg_mae": avg_mae_cerebro_full,
    "sbp_rmse": sbp_rmse_cerebro_full,
    "dbp_rmse": dbp_rmse_cerebro_full,
    "sbp_r2": sbp_r2_cerebro_full,
    "dbp_r2": dbp_r2_cerebro_full
}])

results_cerebro_full.to_csv(
    "results_cerebro_full_ft_sbpdbp_loss_seed42.csv",
    index=False
)

results_cerebro_full

,model,checkpoint,fine_tuning,loss,seed,best_epoch,best_val_loss,lr,weight_decay,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,Cerebro,cerebro_foundation_best,full fine-tuning,Waveform + SBP + DBP loss,42,30,0.304696,0.0001,0.01,5.999414,3.112315,4.555865,8.4863,4.784361,0.674529,0.541247


In [26]:
checkpoint_cerebro_continue = torch.load(
    best_model_path_cerebro_full,
    map_location="cpu"
)

model_cerebro_continue = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
)

model_cerebro_continue.load_state_dict(
    checkpoint_cerebro_continue["model_state_dict"]
)

model_cerebro_continue = model_cerebro_continue.to(device)
model_cerebro_continue.eval()

print("Loaded Cerebro checkpoint to continue training")
print("Previous best epoch:", checkpoint_cerebro_continue["epoch"])
print("Previous best val loss:", checkpoint_cerebro_continue["best_val_loss"])

Loaded Cerebro checkpoint to continue training
Previous best epoch: 30
Previous best val loss: 0.30469589359125837


In [27]:
for param in model_cerebro_continue.parameters():
    param.requires_grad = True

criterion_cerebro_continue = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_cerebro_continue = torch.optim.AdamW(
    model_cerebro_continue.parameters(),
    lr=5e-5,
    weight_decay=1e-2
)

num_epochs_cerebro_continue = 20
best_val_loss_cerebro_continue = checkpoint_cerebro_continue["best_val_loss"]

best_model_path_cerebro_continue = "cerebro_full_ft_sbpdbp_loss_lr5e5_continue_seed42.pth"
history_cerebro_continue = []

for epoch in range(num_epochs_cerebro_continue):
    train_loss = train_one_epoch(
        model_cerebro_continue,
        train_loader,
        optimizer_cerebro_continue,
        criterion_cerebro_continue,
        device
    )

    val_loss = validate_one_epoch(
        model_cerebro_continue,
        val_loader,
        criterion_cerebro_continue,
        device
    )

    history_cerebro_continue.append({
        "method": "Cerebro_full_finetuning_continue_lr5e5_sbpdbp_loss",
        "continue_epoch": epoch + 1,
        "total_epoch": checkpoint_cerebro_continue["epoch"] + epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 5e-5,
        "weight_decay": 1e-2,
        "seed": SEED
    })

    if val_loss < best_val_loss_cerebro_continue:
        best_val_loss_cerebro_continue = val_loss

        torch.save({
            "epoch": checkpoint_cerebro_continue["epoch"] + epoch + 1,
            "continue_epoch": epoch + 1,
            "method": "Cerebro_full_finetuning_continue_lr5e5_sbpdbp_loss",
            "model_state_dict": model_cerebro_continue.state_dict(),
            "optimizer_state_dict": optimizer_cerebro_continue.state_dict(),
            "best_val_loss": float(best_val_loss_cerebro_continue),
            "previous_best_val_loss": checkpoint_cerebro_continue["best_val_loss"],
            "lr": 5e-5,
            "weight_decay": 1e-2,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_cerebro_continue)

        status = "saved"
    else:
        status = ""

    print(
        f"Continue Epoch [{epoch+1}/{num_epochs_cerebro_continue}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Continue Epoch [1/20] | Train Loss: 0.121651 | Val Loss: 0.276320 saved
Continue Epoch [2/20] | Train Loss: 0.105234 | Val Loss: 0.269452 saved
Continue Epoch [3/20] | Train Loss: 0.099798 | Val Loss: 0.272346 
Continue Epoch [4/20] | Train Loss: 0.097410 | Val Loss: 0.265555 saved
Continue Epoch [5/20] | Train Loss: 0.094581 | Val Loss: 0.268892 
Continue Epoch [6/20] | Train Loss: 0.092881 | Val Loss: 0.265779 
Continue Epoch [7/20] | Train Loss: 0.091387 | Val Loss: 0.264049 saved
Continue Epoch [8/20] | Train Loss: 0.088980 | Val Loss: 0.263198 saved
Continue Epoch [9/20] | Train Loss: 0.087439 | Val Loss: 0.259678 saved
Continue Epoch [10/20] | Train Loss: 0.085795 | Val Loss: 0.264950 
Continue Epoch [11/20] | Train Loss: 0.084350 | Val Loss: 0.258899 saved
Continue Epoch [12/20] | Train Loss: 0.083977 | Val Loss: 0.253994 saved
Continue Epoch [13/20] | Train Loss: 0.082389 | Val Loss: 0.254490 
Continue Epoch [14/20] | Train Loss: 0.080416 | Val Loss: 0.256920 
Continue Epoch [1

In [28]:
checkpoint_cerebro_continue_best = torch.load(
    best_model_path_cerebro_continue,
    map_location="cpu"
)

model_cerebro_continue_best = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
)

model_cerebro_continue_best.load_state_dict(
    checkpoint_cerebro_continue_best["model_state_dict"]
)

model_cerebro_continue_best = model_cerebro_continue_best.to(device)
model_cerebro_continue_best.eval()

print("Loaded best continued Cerebro model")
print("Best total epoch:", checkpoint_cerebro_continue_best["epoch"])
print("Best val loss:", checkpoint_cerebro_continue_best["best_val_loss"])
print("LR:", checkpoint_cerebro_continue_best["lr"])

Loaded best continued Cerebro model
Best total epoch: 49
Best val loss: 0.2502099797846917
LR: 5e-05


In [29]:
y_true_cerebro_continue, y_pred_cerebro_continue = evaluate_model_waveform_cerebro(
    model_cerebro_continue_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_cerebro_continue.shape)
print("y_pred:", y_pred_cerebro_continue.shape)

print("true min/max:", y_true_cerebro_continue.min(), y_true_cerebro_continue.max())
print("pred min/max:", y_pred_cerebro_continue.min(), y_pred_cerebro_continue.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 15.75849 207.9194


In [30]:
sbp_true_cerebro_continue = []
dbp_true_cerebro_continue = []
sbp_pred_cerebro_continue = []
dbp_pred_cerebro_continue = []

for true_sig, pred_sig in zip(y_true_cerebro_continue, y_pred_cerebro_continue):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_cerebro_continue.append(sbp_t)
    dbp_true_cerebro_continue.append(dbp_t)
    sbp_pred_cerebro_continue.append(sbp_p)
    dbp_pred_cerebro_continue.append(dbp_p)

sbp_true_cerebro_continue = np.array(sbp_true_cerebro_continue)
dbp_true_cerebro_continue = np.array(dbp_true_cerebro_continue)
sbp_pred_cerebro_continue = np.array(sbp_pred_cerebro_continue)
dbp_pred_cerebro_continue = np.array(dbp_pred_cerebro_continue)

sbp_mae_cerebro_continue = mean_absolute_error(
    sbp_true_cerebro_continue,
    sbp_pred_cerebro_continue
)

dbp_mae_cerebro_continue = mean_absolute_error(
    dbp_true_cerebro_continue,
    dbp_pred_cerebro_continue
)

sbp_rmse_cerebro_continue = np.sqrt(mean_squared_error(
    sbp_true_cerebro_continue,
    sbp_pred_cerebro_continue
))

dbp_rmse_cerebro_continue = np.sqrt(mean_squared_error(
    dbp_true_cerebro_continue,
    dbp_pred_cerebro_continue
))

sbp_r2_cerebro_continue = r2_score(
    sbp_true_cerebro_continue,
    sbp_pred_cerebro_continue
)

dbp_r2_cerebro_continue = r2_score(
    dbp_true_cerebro_continue,
    dbp_pred_cerebro_continue
)

avg_mae_cerebro_continue = (
    sbp_mae_cerebro_continue + dbp_mae_cerebro_continue
) / 2

print("Cerebro Continued Full FT with LR 5e-5 Results")
print("==============================================")
print("Setting: Continued full fine-tuning from best checkpoint")
print("LR: 5e-5")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_cerebro_continue:.3f}")
print(f"RMSE : {sbp_rmse_cerebro_continue:.3f}")
print(f"R2   : {sbp_r2_cerebro_continue:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_cerebro_continue:.3f}")
print(f"RMSE : {dbp_rmse_cerebro_continue:.3f}")
print(f"R2   : {dbp_r2_cerebro_continue:.4f}")

print("\nTable format:")
print(f"{sbp_mae_cerebro_continue:.2f}/{dbp_mae_cerebro_continue:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_cerebro_continue:.3f}")

Cerebro Continued Full FT with LR 5e-5 Results
Setting: Continued full fine-tuning from best checkpoint
LR: 5e-5
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 5.366
RMSE : 7.824
R2   : 0.7234

DBP
MAE  : 2.913
RMSE : 4.647
R2   : 0.5673

Table format:
5.37/2.91

Average MAE:
4.140


In [31]:
results_cerebro_continue = pd.DataFrame([{
    "model": "Cerebro",
    "checkpoint": "cerebro_foundation_best",
    "fine_tuning": "continued full fine-tuning",
    "loss": "Waveform + SBP + DBP loss",
    "seed": SEED,
    "previous_best_val_loss": checkpoint_cerebro_continue_best["previous_best_val_loss"],
    "best_val_loss": checkpoint_cerebro_continue_best["best_val_loss"],
    "best_total_epoch": checkpoint_cerebro_continue_best["epoch"],
    "continue_epoch": checkpoint_cerebro_continue_best["continue_epoch"],
    "lr": checkpoint_cerebro_continue_best["lr"],
    "weight_decay": checkpoint_cerebro_continue_best["weight_decay"],
    "sbp_mae": sbp_mae_cerebro_continue,
    "dbp_mae": dbp_mae_cerebro_continue,
    "avg_mae": avg_mae_cerebro_continue,
    "sbp_rmse": sbp_rmse_cerebro_continue,
    "dbp_rmse": dbp_rmse_cerebro_continue,
    "sbp_r2": sbp_r2_cerebro_continue,
    "dbp_r2": dbp_r2_cerebro_continue
}])

results_cerebro_continue.to_csv(
    "results_cerebro_continue_full_ft_lr5e5_sbpdbp_loss_seed42.csv",
    index=False
)

results_cerebro_continue

,model,checkpoint,fine_tuning,loss,seed,previous_best_val_loss,best_val_loss,best_total_epoch,continue_epoch,lr,weight_decay,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,Cerebro,cerebro_foundation_best,continued full fine-tuning,Waveform + SBP + DBP loss,42,0.304696,0.25021,49,19,0.00005,0.01,5.365685,2.91334,4.139512,7.823759,4.646635,0.723365,0.567279


# Discriminative LR

In [32]:
import gc
import torch

for name in list(globals().keys()):
    if (
        name.startswith("model_cerebro_disclr")
        or name.startswith("optimizer_cerebro_disclr")
        or name.startswith("checkpoint_cerebro_disclr")
    ):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Allocated GB: 0.269
Reserved GB: 0.307


In [33]:
cerebro_pretrained_path = "/kaggle/input/models/ademizhan/cerebro-foundation-best/pytorch/default/1/cerebro_foundation_best.pth"

model_cerebro_disclr = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
).to(device)

pretrained_dict = torch.load(
    cerebro_pretrained_path,
    map_location="cpu"
)

if isinstance(pretrained_dict, dict) and "model_state_dict" in pretrained_dict:
    pretrained_dict = pretrained_dict["model_state_dict"]

model_dict = model_cerebro_disclr.state_dict()
matched_state_cerebro_disclr = {}

for k, v in pretrained_dict.items():
    candidates = [
        k,
        k.replace("module.", ""),
        "model." + k,
        "foundation." + k
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == v.shape:
            matched_state_cerebro_disclr[candidate] = v
            break

missing, unexpected = model_cerebro_disclr.load_state_dict(
    matched_state_cerebro_disclr,
    strict=False
)

model_cerebro_disclr = model_cerebro_disclr.to(device)

print("Loaded tensors:", len(matched_state_cerebro_disclr))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0


In [34]:
for param in model_cerebro_disclr.parameters():
    param.requires_grad = True

head_params = list(model_cerebro_disclr.regression_head.parameters())

head_param_ids = set(id(p) for p in head_params)

backbone_params = [
    p for p in model_cerebro_disclr.parameters()
    if id(p) not in head_param_ids
]

print("Backbone params:", sum(p.numel() for p in backbone_params))
print("Head params:", sum(p.numel() for p in head_params))
print("Total trainable:", sum(p.numel() for p in model_cerebro_disclr.parameters() if p.requires_grad))

Backbone params: 4476736
Head params: 2160625
Total trainable: 6637361


In [35]:
criterion_cerebro_disclr = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_cerebro_disclr = torch.optim.AdamW(
    [
        {
            "params": backbone_params,
            "lr": 2e-5
        },
        {
            "params": head_params,
            "lr": 1e-4
        }
    ],
    weight_decay=1e-2
)

In [36]:
num_epochs_cerebro_disclr = 30
best_val_loss_cerebro_disclr = float("inf")

best_model_path_cerebro_disclr = "cerebro_full_ft_disclr_backbone2e5_head1e4_sbpdbp_seed42.pth"
history_cerebro_disclr = []

for epoch in range(num_epochs_cerebro_disclr):
    train_loss = train_one_epoch(
        model_cerebro_disclr,
        train_loader,
        optimizer_cerebro_disclr,
        criterion_cerebro_disclr,
        device
    )

    val_loss = validate_one_epoch(
        model_cerebro_disclr,
        val_loader,
        criterion_cerebro_disclr,
        device
    )

    history_cerebro_disclr.append({
        "method": "Cerebro_full_ft_discriminative_lr_sbpdbp_loss",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 1e-4,
        "weight_decay": 1e-2,
        "seed": SEED
    })

    if val_loss < best_val_loss_cerebro_disclr:
        best_val_loss_cerebro_disclr = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": "Cerebro_full_ft_discriminative_lr_sbpdbp_loss",
            "model_state_dict": model_cerebro_disclr.state_dict(),
            "optimizer_state_dict": optimizer_cerebro_disclr.state_dict(),
            "best_val_loss": float(best_val_loss_cerebro_disclr),
            "backbone_lr": 2e-5,
            "head_lr": 1e-4,
            "weight_decay": 1e-2,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_cerebro_disclr)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_cerebro_disclr}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Epoch [1/30] | Train Loss: 1.236739 | Val Loss: 1.098427 saved
Epoch [2/30] | Train Loss: 0.973038 | Val Loss: 0.970734 saved
Epoch [3/30] | Train Loss: 0.855280 | Val Loss: 0.889195 saved
Epoch [4/30] | Train Loss: 0.778968 | Val Loss: 0.845797 saved
Epoch [5/30] | Train Loss: 0.718185 | Val Loss: 0.799930 saved
Epoch [6/30] | Train Loss: 0.670311 | Val Loss: 0.791774 saved
Epoch [7/30] | Train Loss: 0.628335 | Val Loss: 0.741076 saved
Epoch [8/30] | Train Loss: 0.589365 | Val Loss: 0.715474 saved
Epoch [9/30] | Train Loss: 0.551534 | Val Loss: 0.683677 saved
Epoch [10/30] | Train Loss: 0.520926 | Val Loss: 0.656001 saved
Epoch [11/30] | Train Loss: 0.491032 | Val Loss: 0.644906 saved
Epoch [12/30] | Train Loss: 0.466674 | Val Loss: 0.617431 saved
Epoch [13/30] | Train Loss: 0.442436 | Val Loss: 0.600935 saved
Epoch [14/30] | Train Loss: 0.424703 | Val Loss: 0.589800 saved
Epoch [15/30] | Train Loss: 0.403699 | Val Loss: 0.572624 saved
Epoch [16/30] | Train Loss: 0.385454 | Val Loss: 

In [37]:
checkpoint_cerebro_disclr_best = torch.load(
    best_model_path_cerebro_disclr,
    map_location="cpu"
)

model_cerebro_disclr_best = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
)

model_cerebro_disclr_best.load_state_dict(
    checkpoint_cerebro_disclr_best["model_state_dict"]
)

model_cerebro_disclr_best = model_cerebro_disclr_best.to(device)
model_cerebro_disclr_best.eval()

print("Loaded best Cerebro discriminative LR model")
print("Best epoch:", checkpoint_cerebro_disclr_best["epoch"])
print("Best val loss:", checkpoint_cerebro_disclr_best["best_val_loss"])
print("Backbone LR:", checkpoint_cerebro_disclr_best["backbone_lr"])
print("Head LR:", checkpoint_cerebro_disclr_best["head_lr"])

Loaded best Cerebro discriminative LR model
Best epoch: 30
Best val loss: 0.4410348836432796
Backbone LR: 2e-05
Head LR: 0.0001


In [38]:
y_true_cerebro_disclr, y_pred_cerebro_disclr = evaluate_model_waveform_cerebro(
    model_cerebro_disclr_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_cerebro_disclr.shape)
print("y_pred:", y_pred_cerebro_disclr.shape)

print("true min/max:", y_true_cerebro_disclr.min(), y_true_cerebro_disclr.max())
print("pred min/max:", y_pred_cerebro_disclr.min(), y_pred_cerebro_disclr.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 24.300894 200.84665


In [39]:
sbp_true_cerebro_disclr = []
dbp_true_cerebro_disclr = []
sbp_pred_cerebro_disclr = []
dbp_pred_cerebro_disclr = []

for true_sig, pred_sig in zip(y_true_cerebro_disclr, y_pred_cerebro_disclr):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_cerebro_disclr.append(sbp_t)
    dbp_true_cerebro_disclr.append(dbp_t)
    sbp_pred_cerebro_disclr.append(sbp_p)
    dbp_pred_cerebro_disclr.append(dbp_p)

sbp_true_cerebro_disclr = np.array(sbp_true_cerebro_disclr)
dbp_true_cerebro_disclr = np.array(dbp_true_cerebro_disclr)
sbp_pred_cerebro_disclr = np.array(sbp_pred_cerebro_disclr)
dbp_pred_cerebro_disclr = np.array(dbp_pred_cerebro_disclr)

sbp_mae_cerebro_disclr = mean_absolute_error(
    sbp_true_cerebro_disclr,
    sbp_pred_cerebro_disclr
)

dbp_mae_cerebro_disclr = mean_absolute_error(
    dbp_true_cerebro_disclr,
    dbp_pred_cerebro_disclr
)

sbp_rmse_cerebro_disclr = np.sqrt(mean_squared_error(
    sbp_true_cerebro_disclr,
    sbp_pred_cerebro_disclr
))

dbp_rmse_cerebro_disclr = np.sqrt(mean_squared_error(
    dbp_true_cerebro_disclr,
    dbp_pred_cerebro_disclr
))

sbp_r2_cerebro_disclr = r2_score(
    sbp_true_cerebro_disclr,
    sbp_pred_cerebro_disclr
)

dbp_r2_cerebro_disclr = r2_score(
    dbp_true_cerebro_disclr,
    dbp_pred_cerebro_disclr
)

avg_mae_cerebro_disclr = (
    sbp_mae_cerebro_disclr + dbp_mae_cerebro_disclr
) / 2

print("Cerebro Full FT with Discriminative LR Results")
print("==============================================")
print("Setting: Full fine-tuning with discriminative LR")
print("Backbone LR: 2e-5")
print("Head LR: 1e-4")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_cerebro_disclr:.3f}")
print(f"RMSE : {sbp_rmse_cerebro_disclr:.3f}")
print(f"R2   : {sbp_r2_cerebro_disclr:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_cerebro_disclr:.3f}")
print(f"RMSE : {dbp_rmse_cerebro_disclr:.3f}")
print(f"R2   : {dbp_r2_cerebro_disclr:.4f}")

print("\nTable format:")
print(f"{sbp_mae_cerebro_disclr:.2f}/{dbp_mae_cerebro_disclr:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_cerebro_disclr:.3f}")

Cerebro Full FT with Discriminative LR Results
Setting: Full fine-tuning with discriminative LR
Backbone LR: 2e-5
Head LR: 1e-4
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 7.191
RMSE : 9.940
R2   : 0.5534

DBP
MAE  : 3.650
RMSE : 5.400
R2   : 0.4156

Table format:
7.19/3.65

Average MAE:
5.420


In [40]:
results_cerebro_disclr = pd.DataFrame([{
    "model": "Cerebro",
    "checkpoint": "cerebro_foundation_best",
    "fine_tuning": "full fine-tuning with discriminative LR",
    "loss": "Waveform + SBP + DBP loss",
    "seed": SEED,
    "best_epoch": checkpoint_cerebro_disclr_best["epoch"],
    "best_val_loss": checkpoint_cerebro_disclr_best["best_val_loss"],
    "backbone_lr": checkpoint_cerebro_disclr_best["backbone_lr"],
    "head_lr": checkpoint_cerebro_disclr_best["head_lr"],
    "weight_decay": checkpoint_cerebro_disclr_best["weight_decay"],
    "sbp_mae": sbp_mae_cerebro_disclr,
    "dbp_mae": dbp_mae_cerebro_disclr,
    "avg_mae": avg_mae_cerebro_disclr,
    "sbp_rmse": sbp_rmse_cerebro_disclr,
    "dbp_rmse": dbp_rmse_cerebro_disclr,
    "sbp_r2": sbp_r2_cerebro_disclr,
    "dbp_r2": dbp_r2_cerebro_disclr
}])

results_cerebro_disclr.to_csv(
    "results_cerebro_full_ft_disclr_sbpdbp_loss_seed42.csv",
    index=False
)

results_cerebro_disclr

,model,checkpoint,fine_tuning,loss,seed,best_epoch,best_val_loss,backbone_lr,head_lr,weight_decay,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,Cerebro,cerebro_foundation_best,full fine-tuning with discriminative LR,Waveform + SBP + DBP loss,42,30,0.441035,0.00002,0.0001,0.01,7.19112,3.649698,5.420409,9.940439,5.400136,0.553433,0.41556


In [41]:
cerebro_compare = pd.concat(
    [
        results_cerebro_full,
        results_cerebro_continue,
        results_cerebro_disclr
    ],
    ignore_index=True
)

cerebro_compare = cerebro_compare[
    [
        "model",
        "fine_tuning",
        "loss",
        "sbp_mae",
        "dbp_mae",
        "avg_mae",
        "sbp_r2",
        "dbp_r2",
        "best_val_loss"
    ]
]

cerebro_compare.sort_values("avg_mae")

,model,fine_tuning,loss,sbp_mae,dbp_mae,avg_mae,sbp_r2,dbp_r2,best_val_loss
1,Cerebro,continued full fine-tuning,Waveform + SBP + DBP loss,5.365685,2.913340,4.139512,0.723365,0.567279,0.250210
0,Cerebro,full fine-tuning,Waveform + SBP + DBP loss,5.999414,3.112315,4.555865,0.674529,0.541247,0.304696
2,Cerebro,full fine-tuning with discriminative LR,Waveform + SBP + DBP loss,7.191120,3.649698,5.420409,0.553433,0.415560,0.441035


# Two stage

In [42]:
import gc
import torch

for name in list(globals().keys()):
    if (
        name.startswith("model_cerebro_twostage")
        or name.startswith("optimizer_cerebro_twostage")
        or name.startswith("checkpoint_cerebro_twostage")
    ):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Allocated GB: 0.383
Reserved GB: 0.42


In [43]:
cerebro_pretrained_path = "/kaggle/input/models/ademizhan/cerebro-foundation-best/pytorch/default/1/cerebro_foundation_best.pth"

model_cerebro_twostage = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
).to(device)

pretrained_dict = torch.load(
    cerebro_pretrained_path,
    map_location="cpu"
)

if isinstance(pretrained_dict, dict) and "model_state_dict" in pretrained_dict:
    pretrained_dict = pretrained_dict["model_state_dict"]

model_dict = model_cerebro_twostage.state_dict()
matched_state_cerebro_twostage = {}

for k, v in pretrained_dict.items():
    candidates = [
        k,
        k.replace("module.", ""),
        "model." + k,
        "foundation." + k
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == v.shape:
            matched_state_cerebro_twostage[candidate] = v
            break

missing, unexpected = model_cerebro_twostage.load_state_dict(
    matched_state_cerebro_twostage,
    strict=False
)

model_cerebro_twostage = model_cerebro_twostage.to(device)

print("Loaded tensors:", len(matched_state_cerebro_twostage))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0


In [44]:
criterion_cerebro_twostage = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

In [45]:
for param in model_cerebro_twostage.parameters():
    param.requires_grad = False

for param in model_cerebro_twostage.regression_head.parameters():
    param.requires_grad = True

print("Trainable parameters after freezing:")
print(sum(p.numel() for p in model_cerebro_twostage.parameters() if p.requires_grad))

optimizer_cerebro_twostage_stage1 = torch.optim.AdamW(
    model_cerebro_twostage.regression_head.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

num_epochs_stage1 = 10
history_cerebro_twostage = []

for epoch in range(num_epochs_stage1):
    train_loss = train_one_epoch(
        model_cerebro_twostage,
        train_loader,
        optimizer_cerebro_twostage_stage1,
        criterion_cerebro_twostage,
        device
    )

    val_loss = validate_one_epoch(
        model_cerebro_twostage,
        val_loader,
        criterion_cerebro_twostage,
        device
    )

    history_cerebro_twostage.append({
        "method": "Cerebro_two_stage_finetuning",
        "stage": 1,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 1e-4,
        "trainable": "regression_head_only",
        "seed": SEED
    })

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

Trainable parameters after freezing:
2160625
Stage 1 Epoch [1/10] | Train Loss: 1.314144 | Val Loss: 1.229771
Stage 1 Epoch [2/10] | Train Loss: 1.192056 | Val Loss: 1.183693
Stage 1 Epoch [3/10] | Train Loss: 1.162818 | Val Loss: 1.203146
Stage 1 Epoch [4/10] | Train Loss: 1.127836 | Val Loss: 1.166905
Stage 1 Epoch [5/10] | Train Loss: 1.104113 | Val Loss: 1.137622
Stage 1 Epoch [6/10] | Train Loss: 1.095695 | Val Loss: 1.131037
Stage 1 Epoch [7/10] | Train Loss: 1.074262 | Val Loss: 1.119674
Stage 1 Epoch [8/10] | Train Loss: 1.068662 | Val Loss: 1.180263
Stage 1 Epoch [9/10] | Train Loss: 1.054342 | Val Loss: 1.227369
Stage 1 Epoch [10/10] | Train Loss: 1.057028 | Val Loss: 1.130661


In [46]:
for param in model_cerebro_twostage.parameters():
    param.requires_grad = True

print("Trainable parameters after unfreezing:")
print(sum(p.numel() for p in model_cerebro_twostage.parameters() if p.requires_grad))

optimizer_cerebro_twostage_stage2 = torch.optim.AdamW(
    model_cerebro_twostage.parameters(),
    lr=5e-5,
    weight_decay=1e-2
)

num_epochs_stage2 = 20
best_val_loss_cerebro_twostage = float("inf")

best_model_path_cerebro_twostage = "cerebro_two_stage_sbpdbp_loss_seed42.pth"

for epoch in range(num_epochs_stage2):
    train_loss = train_one_epoch(
        model_cerebro_twostage,
        train_loader,
        optimizer_cerebro_twostage_stage2,
        criterion_cerebro_twostage,
        device
    )

    val_loss = validate_one_epoch(
        model_cerebro_twostage,
        val_loader,
        criterion_cerebro_twostage,
        device
    )

    total_epoch = num_epochs_stage1 + epoch + 1

    history_cerebro_twostage.append({
        "method": "Cerebro_two_stage_finetuning",
        "stage": 2,
        "epoch": total_epoch,
        "stage2_epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 5e-5,
        "trainable": "full_model",
        "seed": SEED
    })

    if val_loss < best_val_loss_cerebro_twostage:
        best_val_loss_cerebro_twostage = val_loss

        torch.save({
            "epoch": total_epoch,
            "stage2_epoch": epoch + 1,
            "method": "Cerebro_two_stage_finetuning_sbpdbp_loss",
            "model_state_dict": model_cerebro_twostage.state_dict(),
            "optimizer_state_dict": optimizer_cerebro_twostage_stage2.state_dict(),
            "best_val_loss": float(best_val_loss_cerebro_twostage),
            "stage1_epochs": num_epochs_stage1,
            "stage2_epochs": num_epochs_stage2,
            "stage1_lr": 1e-4,
            "stage2_lr": 5e-5,
            "weight_decay": 1e-2,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_cerebro_twostage)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2}] | "
        f"Total Epoch [{total_epoch}/{num_epochs_stage1 + num_epochs_stage2}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Trainable parameters after unfreezing:
6637361
Stage 2 Epoch [1/20] | Total Epoch [11/30] | Train Loss: 0.932331 | Val Loss: 0.944539 saved
Stage 2 Epoch [2/20] | Total Epoch [12/30] | Train Loss: 0.813591 | Val Loss: 0.866491 saved
Stage 2 Epoch [3/20] | Total Epoch [13/30] | Train Loss: 0.730372 | Val Loss: 0.829083 saved
Stage 2 Epoch [4/20] | Total Epoch [14/30] | Train Loss: 0.666058 | Val Loss: 0.769329 saved
Stage 2 Epoch [5/20] | Total Epoch [15/30] | Train Loss: 0.612048 | Val Loss: 0.718548 saved
Stage 2 Epoch [6/20] | Total Epoch [16/30] | Train Loss: 0.559716 | Val Loss: 0.673953 saved
Stage 2 Epoch [7/20] | Total Epoch [17/30] | Train Loss: 0.515674 | Val Loss: 0.643970 saved
Stage 2 Epoch [8/20] | Total Epoch [18/30] | Train Loss: 0.478714 | Val Loss: 0.605094 saved
Stage 2 Epoch [9/20] | Total Epoch [19/30] | Train Loss: 0.442413 | Val Loss: 0.584702 saved
Stage 2 Epoch [10/20] | Total Epoch [20/30] | Train Loss: 0.414404 | Val Loss: 0.556760 saved
Stage 2 Epoch [11/20] 

In [47]:
history_cerebro_twostage_df = pd.DataFrame(history_cerebro_twostage)

history_cerebro_twostage_df.to_csv(
    "history_cerebro_two_stage_sbpdbp_loss_seed42.csv",
    index=False
)

history_cerebro_twostage_df.tail()

,method,stage,epoch,train_loss,val_loss,lr,trainable,seed,stage2_epoch
25,Cerebro_two_stage_finetuning,2,26,0.291887,0.449406,0.00005,full_model,42,16.0
26,Cerebro_two_stage_finetuning,2,27,0.279179,0.440014,0.00005,full_model,42,17.0
27,Cerebro_two_stage_finetuning,2,28,0.266444,0.432119,0.00005,full_model,42,18.0
28,Cerebro_two_stage_finetuning,2,29,0.255934,0.425986,0.00005,full_model,42,19.0
29,Cerebro_two_stage_finetuning,2,30,0.245423,0.413624,0.00005,full_model,42,20.0


In [48]:
checkpoint_cerebro_twostage_best = torch.load(
    best_model_path_cerebro_twostage,
    map_location="cpu"
)

model_cerebro_twostage_best = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
)

model_cerebro_twostage_best.load_state_dict(
    checkpoint_cerebro_twostage_best["model_state_dict"]
)

model_cerebro_twostage_best = model_cerebro_twostage_best.to(device)
model_cerebro_twostage_best.eval()

print("Loaded best Cerebro two-stage model")
print("Best total epoch:", checkpoint_cerebro_twostage_best["epoch"])
print("Best stage 2 epoch:", checkpoint_cerebro_twostage_best["stage2_epoch"])
print("Best val loss:", checkpoint_cerebro_twostage_best["best_val_loss"])

Loaded best Cerebro two-stage model
Best total epoch: 30
Best stage 2 epoch: 20
Best val loss: 0.4136235324490721


In [49]:
y_true_cerebro_twostage, y_pred_cerebro_twostage = evaluate_model_waveform_cerebro(
    model_cerebro_twostage_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_cerebro_twostage.shape)
print("y_pred:", y_pred_cerebro_twostage.shape)

print("true min/max:", y_true_cerebro_twostage.min(), y_true_cerebro_twostage.max())
print("pred min/max:", y_pred_cerebro_twostage.min(), y_pred_cerebro_twostage.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 14.742666 187.06047


In [50]:
sbp_true_cerebro_twostage = []
dbp_true_cerebro_twostage = []
sbp_pred_cerebro_twostage = []
dbp_pred_cerebro_twostage = []

for true_sig, pred_sig in zip(y_true_cerebro_twostage, y_pred_cerebro_twostage):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_cerebro_twostage.append(sbp_t)
    dbp_true_cerebro_twostage.append(dbp_t)
    sbp_pred_cerebro_twostage.append(sbp_p)
    dbp_pred_cerebro_twostage.append(dbp_p)

sbp_true_cerebro_twostage = np.array(sbp_true_cerebro_twostage)
dbp_true_cerebro_twostage = np.array(dbp_true_cerebro_twostage)
sbp_pred_cerebro_twostage = np.array(sbp_pred_cerebro_twostage)
dbp_pred_cerebro_twostage = np.array(dbp_pred_cerebro_twostage)

sbp_mae_cerebro_twostage = mean_absolute_error(
    sbp_true_cerebro_twostage,
    sbp_pred_cerebro_twostage
)

dbp_mae_cerebro_twostage = mean_absolute_error(
    dbp_true_cerebro_twostage,
    dbp_pred_cerebro_twostage
)

sbp_rmse_cerebro_twostage = np.sqrt(mean_squared_error(
    sbp_true_cerebro_twostage,
    sbp_pred_cerebro_twostage
))

dbp_rmse_cerebro_twostage = np.sqrt(mean_squared_error(
    dbp_true_cerebro_twostage,
    dbp_pred_cerebro_twostage
))

sbp_r2_cerebro_twostage = r2_score(
    sbp_true_cerebro_twostage,
    sbp_pred_cerebro_twostage
)

dbp_r2_cerebro_twostage = r2_score(
    dbp_true_cerebro_twostage,
    dbp_pred_cerebro_twostage
)

avg_mae_cerebro_twostage = (
    sbp_mae_cerebro_twostage + dbp_mae_cerebro_twostage
) / 2

print("Cerebro Two-Stage Fine-Tuning Results")
print("====================================")
print("Stage 1: head only, LR 1e-4")
print("Stage 2: full model, LR 5e-5")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_cerebro_twostage:.3f}")
print(f"RMSE : {sbp_rmse_cerebro_twostage:.3f}")
print(f"R2   : {sbp_r2_cerebro_twostage:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_cerebro_twostage:.3f}")
print(f"RMSE : {dbp_rmse_cerebro_twostage:.3f}")
print(f"R2   : {dbp_r2_cerebro_twostage:.4f}")

print("\nTable format:")
print(f"{sbp_mae_cerebro_twostage:.2f}/{dbp_mae_cerebro_twostage:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_cerebro_twostage:.3f}")

Cerebro Two-Stage Fine-Tuning Results
Stage 1: head only, LR 1e-4
Stage 2: full model, LR 5e-5
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 6.746
RMSE : 9.420
R2   : 0.5990

DBP
MAE  : 3.448
RMSE : 5.161
R2   : 0.4662

Table format:
6.75/3.45

Average MAE:
5.097


In [51]:
results_cerebro_twostage = pd.DataFrame([{
    "model": "Cerebro",
    "checkpoint": "cerebro_foundation_best",
    "fine_tuning": "two-stage fine-tuning",
    "loss": "Waveform + SBP + DBP loss",
    "seed": SEED,
    "best_epoch": checkpoint_cerebro_twostage_best["epoch"],
    "best_stage2_epoch": checkpoint_cerebro_twostage_best["stage2_epoch"],
    "best_val_loss": checkpoint_cerebro_twostage_best["best_val_loss"],
    "stage1_lr": checkpoint_cerebro_twostage_best["stage1_lr"],
    "stage2_lr": checkpoint_cerebro_twostage_best["stage2_lr"],
    "weight_decay": checkpoint_cerebro_twostage_best["weight_decay"],
    "sbp_mae": sbp_mae_cerebro_twostage,
    "dbp_mae": dbp_mae_cerebro_twostage,
    "avg_mae": avg_mae_cerebro_twostage,
    "sbp_rmse": sbp_rmse_cerebro_twostage,
    "dbp_rmse": dbp_rmse_cerebro_twostage,
    "sbp_r2": sbp_r2_cerebro_twostage,
    "dbp_r2": dbp_r2_cerebro_twostage
}])

results_cerebro_twostage.to_csv(
    "results_cerebro_two_stage_sbpdbp_loss_seed42.csv",
    index=False
)

results_cerebro_twostage

,model,checkpoint,fine_tuning,loss,seed,best_epoch,best_stage2_epoch,best_val_loss,stage1_lr,stage2_lr,weight_decay,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,Cerebro,cerebro_foundation_best,two-stage fine-tuning,Waveform + SBP + DBP loss,42,30,20,0.413624,0.0001,0.00005,0.01,6.745873,3.448359,5.097116,9.420019,5.160842,0.598968,0.466208


In [52]:
import torch

cerebro_pretrained_path = "/kaggle/input/models/ademizhan/cerebro-foundation-best/pytorch/default/1/cerebro_foundation_best.pth"

checkpoint = torch.load(cerebro_pretrained_path, map_location="cpu")

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print("-", key)
else:
    print("Checkpoint is not a dictionary.")

Checkpoint type: <class 'collections.OrderedDict'>

Checkpoint keys:
- mask_token
- pos_embed
- patch_embed.weight
- patch_embed.bias
- layers.0.attn.qkv.weight
- layers.0.attn.qkv.bias
- layers.0.attn.attn.in_proj_weight
- layers.0.attn.attn.in_proj_bias
- layers.0.attn.attn.out_proj.weight
- layers.0.attn.attn.out_proj.bias
- layers.0.ffn.0.weight
- layers.0.ffn.0.bias
- layers.0.ffn.2.weight
- layers.0.ffn.2.bias
- layers.0.norm1.weight
- layers.0.norm1.bias
- layers.0.norm2.weight
- layers.0.norm2.bias
- layers.1.attn.qkv.weight
- layers.1.attn.qkv.bias
- layers.1.attn.attn.in_proj_weight
- layers.1.attn.attn.in_proj_bias
- layers.1.attn.attn.out_proj.weight
- layers.1.attn.attn.out_proj.bias
- layers.1.ffn.0.weight
- layers.1.ffn.0.bias
- layers.1.ffn.2.weight
- layers.1.ffn.2.bias
- layers.1.norm1.weight
- layers.1.norm1.bias
- layers.1.norm2.weight
- layers.1.norm2.bias
- layers.2.attn.qkv.weight
- layers.2.attn.qkv.bias
- layers.2.attn.attn.in_proj_weight
- layers.2.attn.attn.i

In [53]:
if isinstance(checkpoint, dict):
    metadata_keywords = [
        "data", "dataset", "pretrain", "train", "source",
        "channels", "signal", "modality", "config", "args",
        "epoch", "history", "info"
    ]

    print("Possible metadata fields:\n")

    for key, value in checkpoint.items():
        key_lower = str(key).lower()

        if any(word in key_lower for word in metadata_keywords):
            print("=" * 80)
            print("KEY:", key)
            print("TYPE:", type(value))

            if isinstance(value, (str, int, float, bool)):
                print("VALUE:", value)

            elif isinstance(value, dict):
                print("DICT KEYS:", value.keys())
                for sub_key, sub_value in value.items():
                    print("  ", sub_key, ":", sub_value)

            elif isinstance(value, list):
                print("LIST LENGTH:", len(value))
                print("FIRST ITEMS:", value[:5])

            else:
                print("VALUE:", value)

Possible metadata fields:



In [54]:
state_dict = checkpoint

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]

print("Number of tensors:", len(state_dict))

print("\nFirst 30 tensor names and shapes:")
for i, (name, tensor) in enumerate(state_dict.items()):
    if i >= 30:
        break

    if torch.is_tensor(tensor):
        print(name, tuple(tensor.shape))
    else:
        print(name, type(tensor))

Number of tensors: 118

First 30 tensor names and shapes:
mask_token (1, 1, 192)
pos_embed (1, 18, 192)
patch_embed.weight (192, 64)
patch_embed.bias (192,)
layers.0.attn.qkv.weight (576, 192)
layers.0.attn.qkv.bias (576,)
layers.0.attn.attn.in_proj_weight (576, 192)
layers.0.attn.attn.in_proj_bias (576,)
layers.0.attn.attn.out_proj.weight (192, 192)
layers.0.attn.attn.out_proj.bias (192,)
layers.0.ffn.0.weight (768, 192)
layers.0.ffn.0.bias (768,)
layers.0.ffn.2.weight (192, 768)
layers.0.ffn.2.bias (192,)
layers.0.norm1.weight (192,)
layers.0.norm1.bias (192,)
layers.0.norm2.weight (192,)
layers.0.norm2.bias (192,)
layers.1.attn.qkv.weight (576, 192)
layers.1.attn.qkv.bias (576,)
layers.1.attn.attn.in_proj_weight (576, 192)
layers.1.attn.attn.in_proj_bias (576,)
layers.1.attn.attn.out_proj.weight (192, 192)
layers.1.attn.attn.out_proj.bias (192,)
layers.1.ffn.0.weight (768, 192)
layers.1.ffn.0.bias (768,)
layers.1.ffn.2.weight (192, 768)
layers.1.ffn.2.bias (192,)
layers.1.norm1.weig

In [55]:
search_words = ["patch", "embed", "channel", "input", "conv", "linear", "mask", "recon"]

print("Relevant tensor names:\n")

for name, tensor in state_dict.items():
    name_lower = name.lower()

    if any(word in name_lower for word in search_words):
        if torch.is_tensor(tensor):
            print(name, tuple(tensor.shape))

Relevant tensor names:

mask_token (1, 1, 192)
pos_embed (1, 18, 192)
patch_embed.weight (192, 64)
patch_embed.bias (192,)
recon_head.weight (64, 192)
recon_head.bias (64,)


In [57]:
history_path = "/kaggle/input/datasets/ademizhan/history/pretrain_history.pt"

pretrain_history = torch.load(history_path, map_location="cpu")

print("History type:", type(pretrain_history))

if isinstance(pretrain_history, dict):
    print("\nHistory keys:")
    for key in pretrain_history.keys():
        print("-", key)

    print("\nFull small fields:")
    for key, value in pretrain_history.items():
        if isinstance(value, (str, int, float, bool)):
            print(key, ":", value)
        elif isinstance(value, list):
            print(key, ": list length", len(value))
            print("first items:", value[:3])
        elif isinstance(value, dict):
            print(key, ": dict keys", value.keys())
else:
    print("History length:", len(pretrain_history))
    print("First 5 items:")
    print(pretrain_history[:5])

History type: <class 'dict'>

History keys:
- train_loss
- val_loss
- lrs

Full small fields:
train_loss : list length 100
first items: [1.3131065567334492, 1.520785967508952, 1.52263293052331]
val_loss : list length 100
first items: [1.521033738232866, 1.5179383377485638, 1.6026122524768491]
lrs : list length 100
first items: [0.00041666666666666664, 0.0008333333333333333, 0.00125]


# + more SBP loss

In [58]:
import gc
import torch
import numpy as np
import pandas as pd

gc.collect()
torch.cuda.empty_cache()

best_cerebro_current_path = "cerebro_full_ft_sbpdbp_loss_lr5e5_continue_seed42.pth"

checkpoint_cerebro_sbp08 = torch.load(
    best_cerebro_current_path,
    map_location="cpu"
)

model_cerebro_sbp08 = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
)

model_cerebro_sbp08.load_state_dict(
    checkpoint_cerebro_sbp08["model_state_dict"]
)

model_cerebro_sbp08 = model_cerebro_sbp08.to(device)
model_cerebro_sbp08.eval()

print("Loaded current best Cerebro checkpoint")
print("Previous best epoch:", checkpoint_cerebro_sbp08["epoch"])
print("Previous best val loss:", checkpoint_cerebro_sbp08["best_val_loss"])

Loaded current best Cerebro checkpoint
Previous best epoch: 49
Previous best val loss: 0.2502099797846917


In [59]:
criterion_cerebro_sbp08 = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.8,
    dbp_weight=0.4
)

print("Loss: waveform=1.0, SBP=0.8, DBP=0.4")

Loss: waveform=1.0, SBP=0.8, DBP=0.4


In [60]:
for param in model_cerebro_sbp08.parameters():
    param.requires_grad = True

optimizer_cerebro_sbp08 = torch.optim.AdamW(
    model_cerebro_sbp08.parameters(),
    lr=2e-5,
    weight_decay=1e-2
)

num_epochs_cerebro_sbp08 = 15
best_val_loss_cerebro_sbp08 = float("inf")

best_model_path_cerebro_sbp08 = "cerebro_full_ft_progressive_lr_sbp08_dbp04_seed42.pth"
history_cerebro_sbp08 = []

for epoch in range(num_epochs_cerebro_sbp08):
    train_loss = train_one_epoch(
        model_cerebro_sbp08,
        train_loader,
        optimizer_cerebro_sbp08,
        criterion_cerebro_sbp08,
        device
    )

    val_loss = validate_one_epoch(
        model_cerebro_sbp08,
        val_loader,
        criterion_cerebro_sbp08,
        device
    )

    history_cerebro_sbp08.append({
        "method": "Cerebro_full_ft_progressive_lr_sbp_focused_loss",
        "continue_epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 2e-5,
        "weight_decay": 1e-2,
        "waveform_weight": 1.0,
        "sbp_weight": 0.8,
        "dbp_weight": 0.4,
        "seed": SEED
    })

    if val_loss < best_val_loss_cerebro_sbp08:
        best_val_loss_cerebro_sbp08 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": "Cerebro_full_ft_progressive_lr_sbp_focused_loss",
            "model_state_dict": model_cerebro_sbp08.state_dict(),
            "optimizer_state_dict": optimizer_cerebro_sbp08.state_dict(),
            "best_val_loss": float(best_val_loss_cerebro_sbp08),
            "started_from": best_cerebro_current_path,
            "lr": 2e-5,
            "weight_decay": 1e-2,
            "seed": SEED,
            "waveform_weight": 1.0,
            "sbp_weight": 0.8,
            "dbp_weight": 0.4
        }, best_model_path_cerebro_sbp08)

        status = "saved"
    else:
        status = ""

    print(
        f"SBP-focused Continue Epoch [{epoch+1}/{num_epochs_cerebro_sbp08}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

SBP-focused Continue Epoch [1/15] | Train Loss: 0.071232 | Val Loss: 0.284684 saved
SBP-focused Continue Epoch [2/15] | Train Loss: 0.061674 | Val Loss: 0.283982 saved
SBP-focused Continue Epoch [3/15] | Train Loss: 0.059272 | Val Loss: 0.281696 saved
SBP-focused Continue Epoch [4/15] | Train Loss: 0.058430 | Val Loss: 0.280888 saved
SBP-focused Continue Epoch [5/15] | Train Loss: 0.058340 | Val Loss: 0.281060 
SBP-focused Continue Epoch [6/15] | Train Loss: 0.058329 | Val Loss: 0.281612 
SBP-focused Continue Epoch [7/15] | Train Loss: 0.058504 | Val Loss: 0.278581 saved
SBP-focused Continue Epoch [8/15] | Train Loss: 0.058335 | Val Loss: 0.281162 
SBP-focused Continue Epoch [9/15] | Train Loss: 0.057661 | Val Loss: 0.278595 
SBP-focused Continue Epoch [10/15] | Train Loss: 0.057012 | Val Loss: 0.281987 
SBP-focused Continue Epoch [11/15] | Train Loss: 0.056297 | Val Loss: 0.280091 
SBP-focused Continue Epoch [12/15] | Train Loss: 0.056166 | Val Loss: 0.277150 saved
SBP-focused Continu

In [61]:
history_cerebro_sbp08_df = pd.DataFrame(history_cerebro_sbp08)

history_cerebro_sbp08_df.to_csv(
    "history_cerebro_full_ft_progressive_lr_sbp08_dbp04_seed42.csv",
    index=False
)

history_cerebro_sbp08_df.tail()

,method,continue_epoch,train_loss,val_loss,lr,weight_decay,waveform_weight,sbp_weight,dbp_weight,seed
10,Cerebro_full_ft_progressive_lr_sbp_focused_loss,11,0.056297,0.280091,0.00002,0.01,1.0,0.8,0.4,42
11,Cerebro_full_ft_progressive_lr_sbp_focused_loss,12,0.056166,0.277150,0.00002,0.01,1.0,0.8,0.4,42
12,Cerebro_full_ft_progressive_lr_sbp_focused_loss,13,0.055037,0.279562,0.00002,0.01,1.0,0.8,0.4,42
13,Cerebro_full_ft_progressive_lr_sbp_focused_loss,14,0.054654,0.279676,0.00002,0.01,1.0,0.8,0.4,42
14,Cerebro_full_ft_progressive_lr_sbp_focused_loss,15,0.054128,0.277915,0.00002,0.01,1.0,0.8,0.4,42


In [62]:
checkpoint_cerebro_sbp08_best = torch.load(
    best_model_path_cerebro_sbp08,
    map_location="cpu"
)

model_cerebro_sbp08_best = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
)

model_cerebro_sbp08_best.load_state_dict(
    checkpoint_cerebro_sbp08_best["model_state_dict"]
)

model_cerebro_sbp08_best = model_cerebro_sbp08_best.to(device)
model_cerebro_sbp08_best.eval()

print("Loaded best SBP-focused Cerebro model")
print("Best continue epoch:", checkpoint_cerebro_sbp08_best["epoch"])
print("Best val loss:", checkpoint_cerebro_sbp08_best["best_val_loss"])
print("LR:", checkpoint_cerebro_sbp08_best["lr"])

Loaded best SBP-focused Cerebro model
Best continue epoch: 12
Best val loss: 0.27715011358893793
LR: 2e-05


In [63]:
y_true_cerebro_sbp08, y_pred_cerebro_sbp08 = evaluate_model_waveform_cerebro(
    model_cerebro_sbp08_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_cerebro_sbp08.shape)
print("y_pred:", y_pred_cerebro_sbp08.shape)

print("true min/max:", y_true_cerebro_sbp08.min(), y_true_cerebro_sbp08.max())
print("pred min/max:", y_pred_cerebro_sbp08.min(), y_pred_cerebro_sbp08.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 18.466116 208.0214


In [64]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sbp_true_cerebro_sbp08 = []
dbp_true_cerebro_sbp08 = []
sbp_pred_cerebro_sbp08 = []
dbp_pred_cerebro_sbp08 = []

for true_sig, pred_sig in zip(y_true_cerebro_sbp08, y_pred_cerebro_sbp08):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_cerebro_sbp08.append(sbp_t)
    dbp_true_cerebro_sbp08.append(dbp_t)
    sbp_pred_cerebro_sbp08.append(sbp_p)
    dbp_pred_cerebro_sbp08.append(dbp_p)

sbp_true_cerebro_sbp08 = np.array(sbp_true_cerebro_sbp08)
dbp_true_cerebro_sbp08 = np.array(dbp_true_cerebro_sbp08)
sbp_pred_cerebro_sbp08 = np.array(sbp_pred_cerebro_sbp08)
dbp_pred_cerebro_sbp08 = np.array(dbp_pred_cerebro_sbp08)

sbp_mae_cerebro_sbp08 = mean_absolute_error(
    sbp_true_cerebro_sbp08,
    sbp_pred_cerebro_sbp08
)

dbp_mae_cerebro_sbp08 = mean_absolute_error(
    dbp_true_cerebro_sbp08,
    dbp_pred_cerebro_sbp08
)

sbp_rmse_cerebro_sbp08 = np.sqrt(mean_squared_error(
    sbp_true_cerebro_sbp08,
    sbp_pred_cerebro_sbp08
))

dbp_rmse_cerebro_sbp08 = np.sqrt(mean_squared_error(
    dbp_true_cerebro_sbp08,
    dbp_pred_cerebro_sbp08
))

sbp_r2_cerebro_sbp08 = r2_score(
    sbp_true_cerebro_sbp08,
    sbp_pred_cerebro_sbp08
)

dbp_r2_cerebro_sbp08 = r2_score(
    dbp_true_cerebro_sbp08,
    dbp_pred_cerebro_sbp08
)

avg_mae_cerebro_sbp08 = (
    sbp_mae_cerebro_sbp08 + dbp_mae_cerebro_sbp08
) / 2

print("Cerebro Full FT + Progressive LR Decay + SBP-Focused Loss")
print("=========================================================")
print("Started from: Full FT + LR decay continuation")
print("LR: 2e-5")
print("Loss: waveform=1.0, SBP=0.8, DBP=0.4")

print("\nSBP")
print(f"MAE  : {sbp_mae_cerebro_sbp08:.3f}")
print(f"RMSE : {sbp_rmse_cerebro_sbp08:.3f}")
print(f"R2   : {sbp_r2_cerebro_sbp08:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_cerebro_sbp08:.3f}")
print(f"RMSE : {dbp_rmse_cerebro_sbp08:.3f}")
print(f"R2   : {dbp_r2_cerebro_sbp08:.4f}")

print("\nTable format:")
print(f"{sbp_mae_cerebro_sbp08:.2f}/{dbp_mae_cerebro_sbp08:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_cerebro_sbp08:.3f}")

Cerebro Full FT + Progressive LR Decay + SBP-Focused Loss
Started from: Full FT + LR decay continuation
LR: 2e-5
Loss: waveform=1.0, SBP=0.8, DBP=0.4

SBP
MAE  : 5.178
RMSE : 7.632
R2   : 0.7368

DBP
MAE  : 2.965
RMSE : 4.677
R2   : 0.5615

Table format:
5.18/2.96

Average MAE:
4.071


In [65]:
results_cerebro_sbp08 = pd.DataFrame([{
    "model": "Cerebro",
    "checkpoint": "cerebro_foundation_best",
    "fine_tuning": "full FT + progressive LR decay + SBP-focused loss",
    "started_from": best_cerebro_current_path,
    "loss": "Waveform + 0.8 SBP + 0.4 DBP",
    "seed": SEED,
    "best_continue_epoch": checkpoint_cerebro_sbp08_best["epoch"],
    "best_val_loss": checkpoint_cerebro_sbp08_best["best_val_loss"],
    "lr": checkpoint_cerebro_sbp08_best["lr"],
    "weight_decay": checkpoint_cerebro_sbp08_best["weight_decay"],
    "waveform_weight": checkpoint_cerebro_sbp08_best["waveform_weight"],
    "sbp_weight": checkpoint_cerebro_sbp08_best["sbp_weight"],
    "dbp_weight": checkpoint_cerebro_sbp08_best["dbp_weight"],
    "sbp_mae": sbp_mae_cerebro_sbp08,
    "dbp_mae": dbp_mae_cerebro_sbp08,
    "avg_mae": avg_mae_cerebro_sbp08,
    "sbp_rmse": sbp_rmse_cerebro_sbp08,
    "dbp_rmse": dbp_rmse_cerebro_sbp08,
    "sbp_r2": sbp_r2_cerebro_sbp08,
    "dbp_r2": dbp_r2_cerebro_sbp08
}])

results_cerebro_sbp08.to_csv(
    "results_cerebro_full_ft_progressive_lr_sbp08_dbp04_seed42.csv",
    index=False
)

results_cerebro_sbp08

,model,checkpoint,fine_tuning,started_from,loss,seed,best_continue_epoch,best_val_loss,lr,weight_decay,waveform_weight,sbp_weight,dbp_weight,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,Cerebro,cerebro_foundation_best,full FT + progressive LR decay + SBP-focused loss,cerebro_full_ft_sbpdbp_loss_lr5e5_continue_see...,Waveform + 0.8 SBP + 0.4 DBP,42,12,0.27715,0.00002,0.01,1.0,0.8,0.4,5.178093,2.964501,4.071297,7.631614,4.67733,0.736786,0.561543


# Full FT + progressive LR decay + peak-aware loss

Phase 1: LR = 1e-4, 30 epochs
Phase 2: LR = 5e-5, 20 epochs
Phase 3: LR = 2e-5, 10 epochs
Loss: Peak-aware BP loss

In [66]:
import torch
import torch.nn as nn

class PeakAwareBPLoss(nn.Module):
    def __init__(
        self,
        waveform_weight=1.0,
        sbp_weight=1.0,
        dbp_weight=0.4,
        pulse_pressure_weight=0.25,
        derivative_weight=0.05,
        topk=10
    ):
        super().__init__()
        self.waveform_weight = waveform_weight
        self.sbp_weight = sbp_weight
        self.dbp_weight = dbp_weight
        self.pulse_pressure_weight = pulse_pressure_weight
        self.derivative_weight = derivative_weight
        self.topk = topk
        self.mse = nn.MSELoss()

    def topk_sbp(self, x):
        values = torch.topk(x, k=self.topk, dim=-1).values
        return values.mean(dim=-1)

    def topk_dbp(self, x):
        values = torch.topk(-x, k=self.topk, dim=-1).values
        return -values.mean(dim=-1)

    def forward(self, pred, target):
        if pred.shape != target.shape:
            pred = pred.view_as(target)

        waveform_loss = self.mse(pred, target)

        pred_sbp = self.topk_sbp(pred)
        target_sbp = self.topk_sbp(target)

        pred_dbp = self.topk_dbp(pred)
        target_dbp = self.topk_dbp(target)

        sbp_loss = self.mse(pred_sbp, target_sbp)
        dbp_loss = self.mse(pred_dbp, target_dbp)

        pred_pp = pred_sbp - pred_dbp
        target_pp = target_sbp - target_dbp
        pulse_pressure_loss = self.mse(pred_pp, target_pp)

        pred_deriv = pred[..., 1:] - pred[..., :-1]
        target_deriv = target[..., 1:] - target[..., :-1]
        derivative_loss = self.mse(pred_deriv, target_deriv)

        return (
            self.waveform_weight * waveform_loss
            + self.sbp_weight * sbp_loss
            + self.dbp_weight * dbp_loss
            + self.pulse_pressure_weight * pulse_pressure_loss
            + self.derivative_weight * derivative_loss
        )

In [67]:
import gc
import torch
import numpy as np
import pandas as pd

gc.collect()
torch.cuda.empty_cache()

cerebro_pretrained_path = "/kaggle/input/models/ademizhan/cerebro-foundation-best/pytorch/default/1/cerebro_foundation_best.pth"

model_cerebro_peak_prog = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
).to(device)

pretrained_dict = torch.load(
    cerebro_pretrained_path,
    map_location="cpu"
)

if isinstance(pretrained_dict, dict) and "model_state_dict" in pretrained_dict:
    pretrained_dict = pretrained_dict["model_state_dict"]

model_dict = model_cerebro_peak_prog.state_dict()
matched_state_cerebro_peak_prog = {}

for k, v in pretrained_dict.items():
    candidates = [
        k,
        k.replace("module.", ""),
        "model." + k,
        "foundation." + k
    ]

    for candidate in candidates:
        if candidate in model_dict and model_dict[candidate].shape == v.shape:
            matched_state_cerebro_peak_prog[candidate] = v
            break

missing, unexpected = model_cerebro_peak_prog.load_state_dict(
    matched_state_cerebro_peak_prog,
    strict=False
)

model_cerebro_peak_prog = model_cerebro_peak_prog.to(device)

print("Fresh pretrained Cerebro loaded")
print("Loaded tensors:", len(matched_state_cerebro_peak_prog))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))
print("Trainable parameters:", sum(p.numel() for p in model_cerebro_peak_prog.parameters() if p.requires_grad))

Fresh pretrained Cerebro loaded
Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0
Trainable parameters: 6637361


In [68]:
criterion_cerebro_peak_prog = PeakAwareBPLoss(
    waveform_weight=1.0,
    sbp_weight=1.0,
    dbp_weight=0.4,
    pulse_pressure_weight=0.25,
    derivative_weight=0.05,
    topk=10
)

print("Loss: waveform + top-k SBP + top-k DBP + pulse pressure + derivative")

Loss: waveform + top-k SBP + top-k DBP + pulse pressure + derivative


In [69]:
for param in model_cerebro_peak_prog.parameters():
    param.requires_grad = True

lr_schedule_phases_peak = [
    {
        "phase": 1,
        "lr": 1e-4,
        "epochs": 30
    },
    {
        "phase": 2,
        "lr": 5e-5,
        "epochs": 20
    },
    {
        "phase": 3,
        "lr": 2e-5,
        "epochs": 10
    }
]

best_val_loss_cerebro_peak_prog = float("inf")
best_model_path_cerebro_peak_prog = "cerebro_full_ft_progressive_lr_peak_aware_seed42.pth"
history_cerebro_peak_prog = []

global_epoch = 0
total_epochs = sum(p["epochs"] for p in lr_schedule_phases_peak)

for phase_info in lr_schedule_phases_peak:
    phase = phase_info["phase"]
    lr = phase_info["lr"]
    num_epochs = phase_info["epochs"]

    optimizer_cerebro_peak_prog = torch.optim.AdamW(
        model_cerebro_peak_prog.parameters(),
        lr=lr,
        weight_decay=1e-2
    )

    print(f"\nStarting Phase {phase}: LR = {lr}, Epochs = {num_epochs}")

    for epoch in range(num_epochs):
        global_epoch += 1

        train_loss = train_one_epoch(
            model_cerebro_peak_prog,
            train_loader,
            optimizer_cerebro_peak_prog,
            criterion_cerebro_peak_prog,
            device
        )

        val_loss = validate_one_epoch(
            model_cerebro_peak_prog,
            val_loader,
            criterion_cerebro_peak_prog,
            device
        )

        history_cerebro_peak_prog.append({
            "method": "Cerebro_full_ft_progressive_lr_peak_aware_loss",
            "global_epoch": global_epoch,
            "phase": phase,
            "phase_epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "lr": lr,
            "weight_decay": 1e-2,
            "waveform_weight": 1.0,
            "sbp_weight": 1.0,
            "dbp_weight": 0.4,
            "pulse_pressure_weight": 0.25,
            "derivative_weight": 0.05,
            "topk": 10,
            "seed": SEED
        })

        if val_loss < best_val_loss_cerebro_peak_prog:
            best_val_loss_cerebro_peak_prog = val_loss

            torch.save({
                "global_epoch": global_epoch,
                "phase": phase,
                "phase_epoch": epoch + 1,
                "method": "Cerebro_full_ft_progressive_lr_peak_aware_loss",
                "model_state_dict": model_cerebro_peak_prog.state_dict(),
                "optimizer_state_dict": optimizer_cerebro_peak_prog.state_dict(),
                "best_val_loss": float(best_val_loss_cerebro_peak_prog),
                "lr": lr,
                "weight_decay": 1e-2,
                "seed": SEED,
                "waveform_weight": 1.0,
                "sbp_weight": 1.0,
                "dbp_weight": 0.4,
                "pulse_pressure_weight": 0.25,
                "derivative_weight": 0.05,
                "topk": 10,
                "lr_schedule": lr_schedule_phases_peak
            }, best_model_path_cerebro_peak_prog)

            status = "saved"
        else:
            status = ""

        print(
            f"Global Epoch [{global_epoch}/{total_epochs}] | "
            f"Phase {phase} Epoch [{epoch+1}/{num_epochs}] | "
            f"LR: {lr:.0e} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} {status}"
        )


Starting Phase 1: LR = 0.0001, Epochs = 30
Global Epoch [1/60] | Phase 1 Epoch [1/30] | LR: 1e-04 | Train Loss: 1.541192 | Val Loss: 1.343398 saved
Global Epoch [2/60] | Phase 1 Epoch [2/30] | LR: 1e-04 | Train Loss: 1.101008 | Val Loss: 1.060138 saved
Global Epoch [3/60] | Phase 1 Epoch [3/30] | LR: 1e-04 | Train Loss: 0.861745 | Val Loss: 0.880229 saved
Global Epoch [4/60] | Phase 1 Epoch [4/30] | LR: 1e-04 | Train Loss: 0.698624 | Val Loss: 0.776530 saved
Global Epoch [5/60] | Phase 1 Epoch [5/30] | LR: 1e-04 | Train Loss: 0.588360 | Val Loss: 0.757278 saved
Global Epoch [6/60] | Phase 1 Epoch [6/30] | LR: 1e-04 | Train Loss: 0.514404 | Val Loss: 0.649717 saved
Global Epoch [7/60] | Phase 1 Epoch [7/30] | LR: 1e-04 | Train Loss: 0.457295 | Val Loss: 0.619639 saved
Global Epoch [8/60] | Phase 1 Epoch [8/30] | LR: 1e-04 | Train Loss: 0.406820 | Val Loss: 0.589310 saved
Global Epoch [9/60] | Phase 1 Epoch [9/30] | LR: 1e-04 | Train Loss: 0.370141 | Val Loss: 0.551949 saved
Global Epoc

In [70]:
history_cerebro_peak_prog_df = pd.DataFrame(history_cerebro_peak_prog)

history_cerebro_peak_prog_df.to_csv(
    "history_cerebro_full_ft_progressive_lr_peak_aware_seed42.csv",
    index=False
)

history_cerebro_peak_prog_df.tail()

,method,global_epoch,phase,phase_epoch,train_loss,val_loss,lr,weight_decay,waveform_weight,sbp_weight,dbp_weight,pulse_pressure_weight,derivative_weight,topk,seed
55,Cerebro_full_ft_progressive_lr_peak_aware_loss,56,3,6,0.048580,0.324529,0.00002,0.01,1.0,1.0,0.4,0.25,0.05,10,42
56,Cerebro_full_ft_progressive_lr_peak_aware_loss,57,3,7,0.048038,0.322180,0.00002,0.01,1.0,1.0,0.4,0.25,0.05,10,42
57,Cerebro_full_ft_progressive_lr_peak_aware_loss,58,3,8,0.047372,0.324749,0.00002,0.01,1.0,1.0,0.4,0.25,0.05,10,42
58,Cerebro_full_ft_progressive_lr_peak_aware_loss,59,3,9,0.047094,0.321026,0.00002,0.01,1.0,1.0,0.4,0.25,0.05,10,42
59,Cerebro_full_ft_progressive_lr_peak_aware_loss,60,3,10,0.046524,0.322276,0.00002,0.01,1.0,1.0,0.4,0.25,0.05,10,42


In [71]:
checkpoint_cerebro_peak_prog_best = torch.load(
    best_model_path_cerebro_peak_prog,
    map_location="cpu"
)

model_cerebro_peak_prog_best = CEReBrO_Foundation(
    num_channels=2,
    mode="regression"
)

model_cerebro_peak_prog_best.load_state_dict(
    checkpoint_cerebro_peak_prog_best["model_state_dict"]
)

model_cerebro_peak_prog_best = model_cerebro_peak_prog_best.to(device)
model_cerebro_peak_prog_best.eval()

print("Loaded best Cerebro peak-aware progressive LR model")
print("Best global epoch:", checkpoint_cerebro_peak_prog_best["global_epoch"])
print("Best phase:", checkpoint_cerebro_peak_prog_best["phase"])
print("Best phase epoch:", checkpoint_cerebro_peak_prog_best["phase_epoch"])
print("Best val loss:", checkpoint_cerebro_peak_prog_best["best_val_loss"])
print("Best LR:", checkpoint_cerebro_peak_prog_best["lr"])

Loaded best Cerebro peak-aware progressive LR model
Best global epoch: 59
Best phase: 3
Best phase epoch: 9
Best val loss: 0.32102648514754917
Best LR: 2e-05


In [73]:
y_true_cerebro_peak_prog, y_pred_cerebro_peak_prog = evaluate_model_waveform_cerebro(
    model_cerebro_peak_prog_best,
    test_loader,
    y_scaler,
    device
)

print("y_true:", y_true_cerebro_peak_prog.shape)
print("y_pred:", y_pred_cerebro_peak_prog.shape)

print("true min/max:", y_true_cerebro_peak_prog.min(), y_true_cerebro_peak_prog.max())
print("pred min/max:", y_pred_cerebro_peak_prog.min(), y_pred_cerebro_peak_prog.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: 14.570433 186.14299


In [74]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sbp_true_cerebro_peak_prog = []
dbp_true_cerebro_peak_prog = []
sbp_pred_cerebro_peak_prog = []
dbp_pred_cerebro_peak_prog = []

for true_sig, pred_sig in zip(y_true_cerebro_peak_prog, y_pred_cerebro_peak_prog):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_cerebro_peak_prog.append(sbp_t)
    dbp_true_cerebro_peak_prog.append(dbp_t)
    sbp_pred_cerebro_peak_prog.append(sbp_p)
    dbp_pred_cerebro_peak_prog.append(dbp_p)

sbp_true_cerebro_peak_prog = np.array(sbp_true_cerebro_peak_prog)
dbp_true_cerebro_peak_prog = np.array(dbp_true_cerebro_peak_prog)
sbp_pred_cerebro_peak_prog = np.array(sbp_pred_cerebro_peak_prog)
dbp_pred_cerebro_peak_prog = np.array(dbp_pred_cerebro_peak_prog)

sbp_mae_cerebro_peak_prog = mean_absolute_error(
    sbp_true_cerebro_peak_prog,
    sbp_pred_cerebro_peak_prog
)

dbp_mae_cerebro_peak_prog = mean_absolute_error(
    dbp_true_cerebro_peak_prog,
    dbp_pred_cerebro_peak_prog
)

sbp_rmse_cerebro_peak_prog = np.sqrt(mean_squared_error(
    sbp_true_cerebro_peak_prog,
    sbp_pred_cerebro_peak_prog
))

dbp_rmse_cerebro_peak_prog = np.sqrt(mean_squared_error(
    dbp_true_cerebro_peak_prog,
    dbp_pred_cerebro_peak_prog
))

sbp_r2_cerebro_peak_prog = r2_score(
    sbp_true_cerebro_peak_prog,
    sbp_pred_cerebro_peak_prog
)

dbp_r2_cerebro_peak_prog = r2_score(
    dbp_true_cerebro_peak_prog,
    dbp_pred_cerebro_peak_prog
)

avg_mae_cerebro_peak_prog = (
    sbp_mae_cerebro_peak_prog + dbp_mae_cerebro_peak_prog
) / 2

print("Cerebro Full FT with Progressive LR Decay + Peak-Aware Loss")
print("==========================================================")
print("Method: Full fine-tuning with progressive LR decay")
print("LR schedule: 1e-4 -> 5e-5 -> 2e-5")
print("Loss: waveform + top-k SBP/DBP + pulse pressure + derivative")

print("\nSBP")
print(f"MAE  : {sbp_mae_cerebro_peak_prog:.3f}")
print(f"RMSE : {sbp_rmse_cerebro_peak_prog:.3f}")
print(f"R2   : {sbp_r2_cerebro_peak_prog:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_cerebro_peak_prog:.3f}")
print(f"RMSE : {dbp_rmse_cerebro_peak_prog:.3f}")
print(f"R2   : {dbp_r2_cerebro_peak_prog:.4f}")

print("\nTable format:")
print(f"{sbp_mae_cerebro_peak_prog:.2f}/{dbp_mae_cerebro_peak_prog:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_cerebro_peak_prog:.3f}")

Cerebro Full FT with Progressive LR Decay + Peak-Aware Loss
Method: Full fine-tuning with progressive LR decay
LR schedule: 1e-4 -> 5e-5 -> 2e-5
Loss: waveform + top-k SBP/DBP + pulse pressure + derivative

SBP
MAE  : 5.187
RMSE : 7.438
R2   : 0.7500

DBP
MAE  : 3.176
RMSE : 4.916
R2   : 0.5157

Table format:
5.19/3.18

Average MAE:
4.182


In [75]:
results_cerebro_peak_prog = pd.DataFrame([{
    "model": "Cerebro",
    "checkpoint": "cerebro_foundation_best",
    "fine_tuning": "full fine-tuning with progressive LR decay",
    "loss": "peak-aware BP loss",
    "seed": SEED,
    "lr_schedule": "1e-4_30ep_to_5e-5_20ep_to_2e-5_10ep",
    "best_global_epoch": checkpoint_cerebro_peak_prog_best["global_epoch"],
    "best_phase": checkpoint_cerebro_peak_prog_best["phase"],
    "best_phase_epoch": checkpoint_cerebro_peak_prog_best["phase_epoch"],
    "best_val_loss": checkpoint_cerebro_peak_prog_best["best_val_loss"],
    "best_lr": checkpoint_cerebro_peak_prog_best["lr"],
    "weight_decay": checkpoint_cerebro_peak_prog_best["weight_decay"],
    "waveform_weight": checkpoint_cerebro_peak_prog_best["waveform_weight"],
    "sbp_weight": checkpoint_cerebro_peak_prog_best["sbp_weight"],
    "dbp_weight": checkpoint_cerebro_peak_prog_best["dbp_weight"],
    "pulse_pressure_weight": checkpoint_cerebro_peak_prog_best["pulse_pressure_weight"],
    "derivative_weight": checkpoint_cerebro_peak_prog_best["derivative_weight"],
    "topk": checkpoint_cerebro_peak_prog_best["topk"],
    "sbp_mae": sbp_mae_cerebro_peak_prog,
    "dbp_mae": dbp_mae_cerebro_peak_prog,
    "avg_mae": avg_mae_cerebro_peak_prog,
    "sbp_rmse": sbp_rmse_cerebro_peak_prog,
    "dbp_rmse": dbp_rmse_cerebro_peak_prog,
    "sbp_r2": sbp_r2_cerebro_peak_prog,
    "dbp_r2": dbp_r2_cerebro_peak_prog
}])

results_cerebro_peak_prog.to_csv(
    "results_cerebro_full_ft_progressive_lr_peak_aware_seed42.csv",
    index=False
)

results_cerebro_peak_prog

,model,checkpoint,fine_tuning,loss,seed,lr_schedule,best_global_epoch,best_phase,best_phase_epoch,best_val_loss,...,pulse_pressure_weight,derivative_weight,topk,sbp_mae,dbp_mae,avg_mae,sbp_rmse,dbp_rmse,sbp_r2,dbp_r2
0,Cerebro,cerebro_foundation_best,full fine-tuning with progressive LR decay,peak-aware BP loss,42,1e-4_30ep_to_5e-5_20ep_to_2e-5_10ep,59,3,9,0.321026,...,0.25,0.05,10,5.187251,3.176488,4.181869,7.437879,4.915943,0.74998,0.515667


# with Self-supervised adaptation

In [77]:
import gc
import torch
from torch.utils.data import TensorDataset, DataLoader

gc.collect()
torch.cuda.empty_cache()

def make_ecg_only_tensor(x):
    x = torch.tensor(x, dtype=torch.float32)

    # If shape is [N, 625], make it [N, 1, 625]
    if x.ndim == 2:
        x = x.unsqueeze(1)

    # Keep only ECG channel
    return x[:, :1, :].float()


def make_abp_tensor(y):
    y = torch.tensor(y, dtype=torch.float32)

    # If shape is [N, 625], make it [N, 1, 625]
    if y.ndim == 2:
        y = y.unsqueeze(1)

    return y.float()


X_train_ecg = make_ecg_only_tensor(X_train)
X_val_ecg = make_ecg_only_tensor(X_val)
X_test_ecg = make_ecg_only_tensor(X_test)

y_train_tensor = make_abp_tensor(y_train)
y_val_tensor = make_abp_tensor(y_val)
y_test_tensor = make_abp_tensor(y_test)

print("X_train_ecg:", X_train_ecg.shape)
print("X_val_ecg:", X_val_ecg.shape)
print("X_test_ecg:", X_test_ecg.shape)

print("y_train_tensor:", y_train_tensor.shape)
print("y_val_tensor:", y_val_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)

batch_size = 8   # use 8 for safety on T4; if no OOM, you can try 16

g = torch.Generator()
g.manual_seed(SEED if "SEED" in globals() else seed)

ssl_train_loader = DataLoader(
    TensorDataset(X_train_ecg),
    batch_size=batch_size,
    shuffle=True,
    generator=g
)

ssl_val_loader = DataLoader(
    TensorDataset(X_val_ecg),
    batch_size=batch_size,
    shuffle=False
)

train_loader_ecg = DataLoader(
    TensorDataset(X_train_ecg, y_train_tensor),
    batch_size=batch_size,
    shuffle=True,
    generator=g
)

val_loader_ecg = DataLoader(
    TensorDataset(X_val_ecg, y_val_tensor),
    batch_size=batch_size,
    shuffle=False
)

test_loader_ecg = DataLoader(
    TensorDataset(X_test_ecg, y_test_tensor),
    batch_size=batch_size,
    shuffle=False
)

X_train_ecg: torch.Size([16204, 1, 625])
X_val_ecg: torch.Size([4051, 1, 625])
X_test_ecg: torch.Size([5064, 1, 625])
y_train_tensor: torch.Size([16204, 1, 625])
y_val_tensor: torch.Size([4051, 1, 625])
y_test_tensor: torch.Size([5064, 1, 625])


In [78]:
import torch.nn as nn

class CerebroECGOnlyWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        if x.ndim == 2:
            x = x.unsqueeze(1)

        # ECG-only: keep one channel
        x = x[:, :1, :]

        # Cerebro checkpoint expects 2 channels, so duplicate ECG only
        x = x.repeat(1, 2, 1)

        out = self.base_model(x)

        if out.ndim == 2:
            out = out.unsqueeze(1)

        return out

In [79]:
def load_fresh_cerebro_ecg_wrapper(device):
    cerebro_pretrained_path = "/kaggle/input/models/ademizhan/cerebro-foundation-best/pytorch/default/1/cerebro_foundation_best.pth"

    base_model = CEReBrO_Foundation(
        num_channels=2,
        mode="regression"
    ).to(device)

    pretrained_dict = torch.load(
        cerebro_pretrained_path,
        map_location="cpu"
    )

    if isinstance(pretrained_dict, dict) and "model_state_dict" in pretrained_dict:
        pretrained_dict = pretrained_dict["model_state_dict"]

    model_dict = base_model.state_dict()
    matched_state = {}

    for k, v in pretrained_dict.items():
        candidates = [
            k,
            k.replace("module.", ""),
            "model." + k,
            "foundation." + k
        ]

        for candidate in candidates:
            if candidate in model_dict and model_dict[candidate].shape == v.shape:
                matched_state[candidate] = v
                break

    missing, unexpected = base_model.load_state_dict(
        matched_state,
        strict=False
    )

    model = CerebroECGOnlyWrapper(base_model).to(device)

    print("Fresh pretrained Cerebro loaded")
    print("Loaded tensors:", len(matched_state))
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
    print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

    return model

In [80]:
model_test = load_fresh_cerebro_ecg_wrapper(device)

xb, yb = next(iter(train_loader_ecg))
xb = xb.to(device).float()

with torch.no_grad():
    out = model_test(xb)

print("Input:", xb.shape)
print("Output:", out.shape)
print("Target:", yb.shape)

del model_test
torch.cuda.empty_cache()

Fresh pretrained Cerebro loaded
Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0
Trainable parameters: 6637361
Input: torch.Size([8, 1, 625])
Output: torch.Size([8, 1, 625])
Target: torch.Size([8, 1, 625])


In [81]:
def random_patch_mask_1d(x, mask_ratio=0.4, patch_size=64):
    x_masked = x.clone()
    B, C, L = x.shape

    num_patches = int(np.ceil(L / patch_size))
    patch_mask = torch.rand(B, num_patches, device=x.device) < mask_ratio

    for p in range(num_patches):
        start = p * patch_size
        end = min((p + 1) * patch_size, L)

        mask_b = patch_mask[:, p]

        if mask_b.any():
            x_masked[mask_b, :, start:end] = 0.0

    return x_masked


def train_ecg_reconstruction_epoch(model, loader, optimizer, criterion, device, mask_ratio=0.4):
    model.train()
    total_loss = 0.0

    for batch in loader:
        xb = batch[0].to(device).float()

        xb_masked = random_patch_mask_1d(
            xb,
            mask_ratio=mask_ratio,
            patch_size=64
        )

        optimizer.zero_grad()

        pred = model(xb_masked)

        if pred.shape != xb.shape:
            pred = pred.view_as(xb)

        loss = criterion(pred, xb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


def validate_ecg_reconstruction_epoch(model, loader, criterion, device, mask_ratio=0.4):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            xb = batch[0].to(device).float()

            xb_masked = random_patch_mask_1d(
                xb,
                mask_ratio=mask_ratio,
                patch_size=64
            )

            pred = model(xb_masked)

            if pred.shape != xb.shape:
                pred = pred.view_as(xb)

            loss = criterion(pred, xb)

            total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)

In [82]:
model_cerebro_ecg_domain = load_fresh_cerebro_ecg_wrapper(device)

criterion_ecg_recon = nn.MSELoss()

optimizer_ecg_recon = torch.optim.AdamW(
    model_cerebro_ecg_domain.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

num_epochs_ecg_recon = 15
best_val_loss_ecg_recon = float("inf")

best_model_path_ecg_recon = "cerebro_ecg_domain_adaptive_reconstruction_seed42.pth"
history_ecg_recon = []

for epoch in range(num_epochs_ecg_recon):
    train_loss = train_ecg_reconstruction_epoch(
        model_cerebro_ecg_domain,
        ssl_train_loader,
        optimizer_ecg_recon,
        criterion_ecg_recon,
        device,
        mask_ratio=0.4
    )

    val_loss = validate_ecg_reconstruction_epoch(
        model_cerebro_ecg_domain,
        ssl_val_loader,
        criterion_ecg_recon,
        device,
        mask_ratio=0.4
    )

    history_ecg_recon.append({
        "method": "ECG_domain_adaptive_masked_reconstruction",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 1e-4,
        "mask_ratio": 0.4,
        "patch_size": 64,
        "seed": SEED
    })

    if val_loss < best_val_loss_ecg_recon:
        best_val_loss_ecg_recon = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": "ECG_domain_adaptive_masked_reconstruction",
            "model_state_dict": model_cerebro_ecg_domain.state_dict(),
            "optimizer_state_dict": optimizer_ecg_recon.state_dict(),
            "best_val_loss": float(best_val_loss_ecg_recon),
            "lr": 1e-4,
            "mask_ratio": 0.4,
            "patch_size": 64,
            "seed": SEED
        }, best_model_path_ecg_recon)

        status = "saved"
    else:
        status = ""

    print(
        f"ECG Reconstruction Epoch [{epoch+1}/{num_epochs_ecg_recon}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

Fresh pretrained Cerebro loaded
Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0
Trainable parameters: 6637361
ECG Reconstruction Epoch [1/15] | Train Loss: 0.259399 | Val Loss: 0.205226 saved
ECG Reconstruction Epoch [2/15] | Train Loss: 0.187703 | Val Loss: 0.182696 saved
ECG Reconstruction Epoch [3/15] | Train Loss: 0.172358 | Val Loss: 0.171137 saved
ECG Reconstruction Epoch [4/15] | Train Loss: 0.165397 | Val Loss: 0.165667 saved
ECG Reconstruction Epoch [5/15] | Train Loss: 0.156800 | Val Loss: 0.159745 saved
ECG Reconstruction Epoch [6/15] | Train Loss: 0.151641 | Val Loss: 0.153749 saved
ECG Reconstruction Epoch [7/15] | Train Loss: 0.145920 | Val Loss: 0.147822 saved
ECG Reconstruction Epoch [8/15] | Train Loss: 0.139806 | Val Loss: 0.145865 saved
ECG Reconstruction Epoch [9/15] | Train Loss: 0.135832 | Val Loss: 0.140311 saved
ECG Reconstruction Epoch [10/15] | Train Loss: 0.129890 | Val Loss: 0.137710 saved
ECG Reconstruction Epoch [11/15] | Train Loss: 0.126456 | Val 

In [83]:
checkpoint_ecg_recon_best = torch.load(
    best_model_path_ecg_recon,
    map_location="cpu"
)

model_cerebro_ecg_bp = load_fresh_cerebro_ecg_wrapper(device)

model_cerebro_ecg_bp.load_state_dict(
    checkpoint_ecg_recon_best["model_state_dict"]
)

model_cerebro_ecg_bp = model_cerebro_ecg_bp.to(device)
model_cerebro_ecg_bp.eval()

print("Loaded ECG-domain adapted Cerebro")
print("Best reconstruction epoch:", checkpoint_ecg_recon_best["epoch"])
print("Best reconstruction val loss:", checkpoint_ecg_recon_best["best_val_loss"])

Fresh pretrained Cerebro loaded
Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0
Trainable parameters: 6637361
Loaded ECG-domain adapted Cerebro
Best reconstruction epoch: 14
Best reconstruction val loss: 0.12423815953602058


In [84]:
criterion_cerebro_ecg_bp = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

print("Supervised loss: waveform=1.0, SBP=0.5, DBP=0.5")

Supervised loss: waveform=1.0, SBP=0.5, DBP=0.5


In [86]:
for param in model_cerebro_ecg_bp.parameters():
    param.requires_grad = True

lr_schedule_phases_ecg_bp = [
    {"phase": 1, "lr": 1e-4, "epochs": 30},
    {"phase": 2, "lr": 5e-5, "epochs": 20},
    {"phase": 3, "lr": 2e-5, "epochs": 10}
]

best_val_loss_cerebro_ecg_bp = float("inf")
best_model_path_cerebro_ecg_bp = "cerebro_ecg_domain_adapted_progressive_ft_seed42.pth"

history_cerebro_ecg_bp = []
global_epoch = 0
total_epochs = sum(p["epochs"] for p in lr_schedule_phases_ecg_bp)

for phase_info in lr_schedule_phases_ecg_bp:
    phase = phase_info["phase"]
    lr = phase_info["lr"]
    num_epochs = phase_info["epochs"]

    optimizer_cerebro_ecg_bp = torch.optim.AdamW(
        model_cerebro_ecg_bp.parameters(),
        lr=lr,
        weight_decay=1e-2
    )

    print(f"\nStarting supervised Phase {phase}: LR = {lr}, Epochs = {num_epochs}")

    for epoch in range(num_epochs):
        global_epoch += 1

        train_loss = train_one_epoch(
            model_cerebro_ecg_bp,
            train_loader_ecg,
            optimizer_cerebro_ecg_bp,
            criterion_cerebro_ecg_bp,
            device
        )

        val_loss = validate_one_epoch(
            model_cerebro_ecg_bp,
            val_loader_ecg,
            criterion_cerebro_ecg_bp,
            device
        )

        history_cerebro_ecg_bp.append({
            "method": "Cerebro_ECG_domain_adaptation_then_progressive_full_FT",
            "global_epoch": global_epoch,
            "phase": phase,
            "phase_epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "lr": lr,
            "weight_decay": 1e-2,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5,
            "seed": SEED if "SEED" in globals() else seed
        })

        if val_loss < best_val_loss_cerebro_ecg_bp:
            best_val_loss_cerebro_ecg_bp = val_loss

            torch.save({
                "global_epoch": global_epoch,
                "phase": phase,
                "phase_epoch": epoch + 1,
                "method": "Cerebro_ECG_domain_adaptation_then_progressive_full_FT",
                "model_state_dict": model_cerebro_ecg_bp.state_dict(),
                "optimizer_state_dict": optimizer_cerebro_ecg_bp.state_dict(),
                "best_val_loss": float(best_val_loss_cerebro_ecg_bp),
                "started_from": best_model_path_ecg_recon,
                "lr": lr,
                "weight_decay": 1e-2,
                "seed": SEED if "SEED" in globals() else seed,
                "waveform_weight": 1.0,
                "sbp_weight": 0.5,
                "dbp_weight": 0.5,
                "lr_schedule": lr_schedule_phases_ecg_bp
            }, best_model_path_cerebro_ecg_bp)

            status = "saved"
        else:
            status = ""

        print(
            f"Global Epoch [{global_epoch}/{total_epochs}] | "
            f"Phase {phase} Epoch [{epoch+1}/{num_epochs}] | "
            f"LR: {lr:.0e} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} {status}"
        )


Starting supervised Phase 1: LR = 0.0001, Epochs = 30
Global Epoch [1/60] | Phase 1 Epoch [1/30] | LR: 1e-04 | Train Loss: 1.259885 | Val Loss: 0.872308 saved
Global Epoch [2/60] | Phase 1 Epoch [2/30] | LR: 1e-04 | Train Loss: 0.812351 | Val Loss: 0.707779 saved
Global Epoch [3/60] | Phase 1 Epoch [3/30] | LR: 1e-04 | Train Loss: 0.615393 | Val Loss: 0.616739 saved
Global Epoch [4/60] | Phase 1 Epoch [4/30] | LR: 1e-04 | Train Loss: 0.507475 | Val Loss: 0.578817 saved
Global Epoch [5/60] | Phase 1 Epoch [5/30] | LR: 1e-04 | Train Loss: 0.438989 | Val Loss: 0.531155 saved
Global Epoch [6/60] | Phase 1 Epoch [6/30] | LR: 1e-04 | Train Loss: 0.396162 | Val Loss: 0.479148 saved
Global Epoch [7/60] | Phase 1 Epoch [7/30] | LR: 1e-04 | Train Loss: 0.363551 | Val Loss: 0.473520 saved
Global Epoch [8/60] | Phase 1 Epoch [8/30] | LR: 1e-04 | Train Loss: 0.337483 | Val Loss: 0.442190 saved
Global Epoch [9/60] | Phase 1 Epoch [9/30] | LR: 1e-04 | Train Loss: 0.313139 | Val Loss: 0.428714 saved


In [87]:
history_cerebro_ecg_bp_df = pd.DataFrame(history_cerebro_ecg_bp)

history_cerebro_ecg_bp_df.to_csv(
    "history_cerebro_ecg_domain_adapted_progressive_ft_seed42.csv",
    index=False
)

history_cerebro_ecg_bp_df.tail()

,method,global_epoch,phase,phase_epoch,train_loss,val_loss,lr,weight_decay,waveform_weight,sbp_weight,dbp_weight,seed
55,Cerebro_ECG_domain_adaptation_then_progressive...,56,3,6,0.039065,0.261819,0.00002,0.01,1.0,0.5,0.5,42
56,Cerebro_ECG_domain_adaptation_then_progressive...,57,3,7,0.038410,0.261539,0.00002,0.01,1.0,0.5,0.5,42
57,Cerebro_ECG_domain_adaptation_then_progressive...,58,3,8,0.037512,0.259698,0.00002,0.01,1.0,0.5,0.5,42
58,Cerebro_ECG_domain_adaptation_then_progressive...,59,3,9,0.036995,0.261559,0.00002,0.01,1.0,0.5,0.5,42
59,Cerebro_ECG_domain_adaptation_then_progressive...,60,3,10,0.036812,0.261056,0.00002,0.01,1.0,0.5,0.5,42


In [88]:
checkpoint_cerebro_ecg_bp_best = torch.load(
    best_model_path_cerebro_ecg_bp,
    map_location="cpu"
)

model_cerebro_ecg_bp_best = load_fresh_cerebro_ecg_wrapper(device)

model_cerebro_ecg_bp_best.load_state_dict(
    checkpoint_cerebro_ecg_bp_best["model_state_dict"]
)

model_cerebro_ecg_bp_best = model_cerebro_ecg_bp_best.to(device)
model_cerebro_ecg_bp_best.eval()

print("Loaded best ECG-domain adapted Cerebro BP model")
print("Best global epoch:", checkpoint_cerebro_ecg_bp_best["global_epoch"])
print("Best phase:", checkpoint_cerebro_ecg_bp_best["phase"])
print("Best phase epoch:", checkpoint_cerebro_ecg_bp_best["phase_epoch"])
print("Best val loss:", checkpoint_cerebro_ecg_bp_best["best_val_loss"])
print("Best LR:", checkpoint_cerebro_ecg_bp_best["lr"])

Fresh pretrained Cerebro loaded
Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0
Trainable parameters: 6637361
Loaded best ECG-domain adapted Cerebro BP model
Best global epoch: 58
Best phase: 3
Best phase epoch: 8
Best val loss: 0.2596984195528252
Best LR: 2e-05


In [89]:
y_true_cerebro_ecg_bp, y_pred_cerebro_ecg_bp = evaluate_model_waveform_cerebro(
    model_cerebro_ecg_bp_best,
    test_loader_ecg,
    y_scaler,
    device
)

print("y_true:", y_true_cerebro_ecg_bp.shape)
print("y_pred:", y_pred_cerebro_ecg_bp.shape)

print("true min/max:", y_true_cerebro_ecg_bp.min(), y_true_cerebro_ecg_bp.max())
print("pred min/max:", y_pred_cerebro_ecg_bp.min(), y_pred_cerebro_ecg_bp.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: -54.1941 288.42978


In [90]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sbp_true_cerebro_ecg_bp = []
dbp_true_cerebro_ecg_bp = []
sbp_pred_cerebro_ecg_bp = []
dbp_pred_cerebro_ecg_bp = []

for true_sig, pred_sig in zip(y_true_cerebro_ecg_bp, y_pred_cerebro_ecg_bp):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_cerebro_ecg_bp.append(sbp_t)
    dbp_true_cerebro_ecg_bp.append(dbp_t)
    sbp_pred_cerebro_ecg_bp.append(sbp_p)
    dbp_pred_cerebro_ecg_bp.append(dbp_p)

sbp_true_cerebro_ecg_bp = np.array(sbp_true_cerebro_ecg_bp)
dbp_true_cerebro_ecg_bp = np.array(dbp_true_cerebro_ecg_bp)
sbp_pred_cerebro_ecg_bp = np.array(sbp_pred_cerebro_ecg_bp)
dbp_pred_cerebro_ecg_bp = np.array(dbp_pred_cerebro_ecg_bp)

sbp_mae_cerebro_ecg_bp = mean_absolute_error(
    sbp_true_cerebro_ecg_bp,
    sbp_pred_cerebro_ecg_bp
)

dbp_mae_cerebro_ecg_bp = mean_absolute_error(
    dbp_true_cerebro_ecg_bp,
    dbp_pred_cerebro_ecg_bp
)

sbp_rmse_cerebro_ecg_bp = np.sqrt(mean_squared_error(
    sbp_true_cerebro_ecg_bp,
    sbp_pred_cerebro_ecg_bp
))

dbp_rmse_cerebro_ecg_bp = np.sqrt(mean_squared_error(
    dbp_true_cerebro_ecg_bp,
    dbp_pred_cerebro_ecg_bp
))

sbp_r2_cerebro_ecg_bp = r2_score(
    sbp_true_cerebro_ecg_bp,
    sbp_pred_cerebro_ecg_bp
)

dbp_r2_cerebro_ecg_bp = r2_score(
    dbp_true_cerebro_ecg_bp,
    dbp_pred_cerebro_ecg_bp
)

avg_mae_cerebro_ecg_bp = (
    sbp_mae_cerebro_ecg_bp + dbp_mae_cerebro_ecg_bp
) / 2

print("Cerebro ECG-Only Domain Adaptation + Progressive Full FT Results")
print("================================================================")
print("Stage A: ECG-only masked reconstruction adaptation")
print("Stage B: Full fine-tuning with progressive LR decay")
print("LR schedule: 1e-4 -> 5e-5 -> 2e-5")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_cerebro_ecg_bp:.3f}")
print(f"RMSE : {sbp_rmse_cerebro_ecg_bp:.3f}")
print(f"R2   : {sbp_r2_cerebro_ecg_bp:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_cerebro_ecg_bp:.3f}")
print(f"RMSE : {dbp_rmse_cerebro_ecg_bp:.3f}")
print(f"R2   : {dbp_r2_cerebro_ecg_bp:.4f}")

print("\nTable format:")
print(f"{sbp_mae_cerebro_ecg_bp:.2f}/{dbp_mae_cerebro_ecg_bp:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_cerebro_ecg_bp:.3f}")

Cerebro ECG-Only Domain Adaptation + Progressive Full FT Results
Stage A: ECG-only masked reconstruction adaptation
Stage B: Full fine-tuning with progressive LR decay
LR schedule: 1e-4 -> 5e-5 -> 2e-5
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 5.809
RMSE : 8.858
R2   : 0.6454

DBP
MAE  : 2.871
RMSE : 4.974
R2   : 0.5041

Table format:
5.81/2.87

Average MAE:
4.340


# adapt + full

In [92]:
import gc
import torch
import pandas as pd

gc.collect()
torch.cuda.empty_cache()

best_model_path_ecg_recon = "cerebro_ecg_domain_adaptive_reconstruction_seed42.pth"

checkpoint_ecg_recon_best = torch.load(
    best_model_path_ecg_recon,
    map_location="cpu"
)

print("Loaded ECG-domain adaptation checkpoint")
print("Best ECG reconstruction epoch:", checkpoint_ecg_recon_best.get("epoch", None))
print("Best ECG reconstruction val loss:", checkpoint_ecg_recon_best.get("best_val_loss", None))

Loaded ECG-domain adaptation checkpoint
Best ECG reconstruction epoch: 14
Best ECG reconstruction val loss: 0.12423815953602058


In [93]:
model_cerebro_ecg_ft30 = load_fresh_cerebro_ecg_wrapper(device)

missing, unexpected = model_cerebro_ecg_ft30.load_state_dict(
    checkpoint_ecg_recon_best["model_state_dict"],
    strict=False
)

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

model_cerebro_ecg_ft30 = model_cerebro_ecg_ft30.to(device)

for param in model_cerebro_ecg_ft30.parameters():
    param.requires_grad = True

print("Model ready for full fine-tuning")
print("Trainable parameters:", sum(p.numel() for p in model_cerebro_ecg_ft30.parameters() if p.requires_grad))

Fresh pretrained Cerebro loaded
Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0
Trainable parameters: 6637361
Missing keys: 0
Unexpected keys: 0
Model ready for full fine-tuning
Trainable parameters: 6637361


In [98]:
criterion_cerebro_ecg_ft30 = WaveformSBPDBPLoss(
    waveform_weight=1.0,
    sbp_weight=0.5,
    dbp_weight=0.5
)

optimizer_cerebro_ecg_ft30 = torch.optim.AdamW(
    model_cerebro_ecg_ft30.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

num_epochs_cerebro_ecg_ft30 = 30
best_val_loss_cerebro_ecg_ft30 = float("inf")

best_model_path_cerebro_ecg_ft30 = "cerebro_ecg_domain_adapted_full_ft_30ep_lr1e4_seed42.pth"

history_cerebro_ecg_ft30 = []

print("Loss: Waveform + SBP/DBP")
print("LR: 1e-4")
print("Epochs:", num_epochs_cerebro_ecg_ft30)

Loss: Waveform + SBP/DBP
LR: 1e-4
Epochs: 30


In [96]:
print(train_loader_ecg.batch_size)

8


In [97]:
from torch.utils.data import TensorDataset, DataLoader
import torch

batch_size = 32

g = torch.Generator()
g.manual_seed(SEED if "SEED" in globals() else seed)

train_loader_ecg = DataLoader(
    TensorDataset(X_train_ecg, y_train_tensor),
    batch_size=batch_size,
    shuffle=True,
    generator=g
)

val_loader_ecg = DataLoader(
    TensorDataset(X_val_ecg, y_val_tensor),
    batch_size=batch_size,
    shuffle=False
)

test_loader_ecg = DataLoader(
    TensorDataset(X_test_ecg, y_test_tensor),
    batch_size=batch_size,
    shuffle=False
)

print("train_loader_ecg batch size:", train_loader_ecg.batch_size)
print("val_loader_ecg batch size:", val_loader_ecg.batch_size)
print("test_loader_ecg batch size:", test_loader_ecg.batch_size)

train_loader_ecg batch size: 32
val_loader_ecg batch size: 32
test_loader_ecg batch size: 32


In [99]:
for epoch in range(num_epochs_cerebro_ecg_ft30):
    train_loss = train_one_epoch(
        model_cerebro_ecg_ft30,
        train_loader_ecg,
        optimizer_cerebro_ecg_ft30,
        criterion_cerebro_ecg_ft30,
        device
    )

    val_loss = validate_one_epoch(
        model_cerebro_ecg_ft30,
        val_loader_ecg,
        criterion_cerebro_ecg_ft30,
        device
    )

    history_cerebro_ecg_ft30.append({
        "method": "Cerebro_ECG_domain_adaptation_then_full_FT_30ep",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "lr": 1e-4,
        "weight_decay": 1e-2,
        "waveform_weight": 1.0,
        "sbp_weight": 0.5,
        "dbp_weight": 0.5,
        "started_from": best_model_path_ecg_recon,
        "seed": SEED if "SEED" in globals() else seed
    })

    if val_loss < best_val_loss_cerebro_ecg_ft30:
        best_val_loss_cerebro_ecg_ft30 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": "Cerebro_ECG_domain_adaptation_then_full_FT_30ep",
            "model_state_dict": model_cerebro_ecg_ft30.state_dict(),
            "optimizer_state_dict": optimizer_cerebro_ecg_ft30.state_dict(),
            "best_val_loss": float(best_val_loss_cerebro_ecg_ft30),
            "started_from": best_model_path_ecg_recon,
            "lr": 1e-4,
            "weight_decay": 1e-2,
            "seed": SEED if "SEED" in globals() else seed,
            "waveform_weight": 1.0,
            "sbp_weight": 0.5,
            "dbp_weight": 0.5
        }, best_model_path_cerebro_ecg_ft30)

        status = "saved"
    else:
        status = ""

    print(
        f"ECG-adapted Full FT Epoch [{epoch+1}/{num_epochs_cerebro_ecg_ft30}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} {status}"
    )

ECG-adapted Full FT Epoch [1/30] | Train Loss: 0.302267 | Val Loss: 0.410557 saved
ECG-adapted Full FT Epoch [2/30] | Train Loss: 0.258450 | Val Loss: 0.397668 saved
ECG-adapted Full FT Epoch [3/30] | Train Loss: 0.233638 | Val Loss: 0.382890 saved
ECG-adapted Full FT Epoch [4/30] | Train Loss: 0.217344 | Val Loss: 0.378528 saved
ECG-adapted Full FT Epoch [5/30] | Train Loss: 0.202379 | Val Loss: 0.368278 saved
ECG-adapted Full FT Epoch [6/30] | Train Loss: 0.190068 | Val Loss: 0.367907 saved
ECG-adapted Full FT Epoch [7/30] | Train Loss: 0.181382 | Val Loss: 0.357361 saved
ECG-adapted Full FT Epoch [8/30] | Train Loss: 0.172844 | Val Loss: 0.348287 saved
ECG-adapted Full FT Epoch [9/30] | Train Loss: 0.165935 | Val Loss: 0.342071 saved
ECG-adapted Full FT Epoch [10/30] | Train Loss: 0.158180 | Val Loss: 0.357101 
ECG-adapted Full FT Epoch [11/30] | Train Loss: 0.152675 | Val Loss: 0.339779 saved
ECG-adapted Full FT Epoch [12/30] | Train Loss: 0.148243 | Val Loss: 0.337912 saved
ECG-ad

In [100]:

history_cerebro_ecg_ft30_df = pd.DataFrame(history_cerebro_ecg_ft30)

history_cerebro_ecg_ft30_df.to_csv(
    "history_cerebro_ecg_domain_adapted_full_ft_30ep_lr1e4_seed42.csv",
    index=False
)

history_cerebro_ecg_ft30_df.tail()

,method,epoch,train_loss,val_loss,lr,weight_decay,waveform_weight,sbp_weight,dbp_weight,started_from,seed
25,Cerebro_ECG_domain_adaptation_then_full_FT_30ep,26,0.107664,0.315199,0.0001,0.01,1.0,0.5,0.5,cerebro_ecg_domain_adaptive_reconstruction_see...,42
26,Cerebro_ECG_domain_adaptation_then_full_FT_30ep,27,0.104816,0.309997,0.0001,0.01,1.0,0.5,0.5,cerebro_ecg_domain_adaptive_reconstruction_see...,42
27,Cerebro_ECG_domain_adaptation_then_full_FT_30ep,28,0.102687,0.304468,0.0001,0.01,1.0,0.5,0.5,cerebro_ecg_domain_adaptive_reconstruction_see...,42
28,Cerebro_ECG_domain_adaptation_then_full_FT_30ep,29,0.101799,0.308552,0.0001,0.01,1.0,0.5,0.5,cerebro_ecg_domain_adaptive_reconstruction_see...,42
29,Cerebro_ECG_domain_adaptation_then_full_FT_30ep,30,0.100446,0.303125,0.0001,0.01,1.0,0.5,0.5,cerebro_ecg_domain_adaptive_reconstruction_see...,42


In [101]:
import gc
import torch
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

gc.collect()
torch.cuda.empty_cache()

checkpoint_cerebro_ecg_ft30_best = torch.load(
    best_model_path_cerebro_ecg_ft30,
    map_location="cpu"
)

model_cerebro_ecg_ft30_best = load_fresh_cerebro_ecg_wrapper(device)

missing, unexpected = model_cerebro_ecg_ft30_best.load_state_dict(
    checkpoint_cerebro_ecg_ft30_best["model_state_dict"],
    strict=False
)

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

model_cerebro_ecg_ft30_best = model_cerebro_ecg_ft30_best.to(device)
model_cerebro_ecg_ft30_best.eval()

print("Loaded best ECG-adapted Full FT model")
print("Best epoch:", checkpoint_cerebro_ecg_ft30_best["epoch"])
print("Best val loss:", checkpoint_cerebro_ecg_ft30_best["best_val_loss"])

Fresh pretrained Cerebro loaded
Loaded tensors: 118
Missing keys: 2
Unexpected keys: 0
Trainable parameters: 6637361
Missing keys: 0
Unexpected keys: 0
Loaded best ECG-adapted Full FT model
Best epoch: 30
Best val loss: 0.30312482948672237


In [102]:
y_true_cerebro_ecg_ft30, y_pred_cerebro_ecg_ft30 = evaluate_model_waveform_cerebro(
    model_cerebro_ecg_ft30_best,
    test_loader_ecg,
    y_scaler,
    device
)

print("y_true:", y_true_cerebro_ecg_ft30.shape)
print("y_pred:", y_pred_cerebro_ecg_ft30.shape)

print("true min/max:", y_true_cerebro_ecg_ft30.min(), y_true_cerebro_ecg_ft30.max())
print("pred min/max:", y_pred_cerebro_ecg_ft30.min(), y_pred_cerebro_ecg_ft30.max())

y_true: (5064, 625)
y_pred: (5064, 625)
true min/max: 60.0 179.5625
pred min/max: -66.68447 280.98422


In [103]:
sbp_true_cerebro_ecg_ft30 = []
dbp_true_cerebro_ecg_ft30 = []
sbp_pred_cerebro_ecg_ft30 = []
dbp_pred_cerebro_ecg_ft30 = []

for true_sig, pred_sig in zip(y_true_cerebro_ecg_ft30, y_pred_cerebro_ecg_ft30):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_cerebro_ecg_ft30.append(sbp_t)
    dbp_true_cerebro_ecg_ft30.append(dbp_t)
    sbp_pred_cerebro_ecg_ft30.append(sbp_p)
    dbp_pred_cerebro_ecg_ft30.append(dbp_p)

sbp_true_cerebro_ecg_ft30 = np.array(sbp_true_cerebro_ecg_ft30)
dbp_true_cerebro_ecg_ft30 = np.array(dbp_true_cerebro_ecg_ft30)
sbp_pred_cerebro_ecg_ft30 = np.array(sbp_pred_cerebro_ecg_ft30)
dbp_pred_cerebro_ecg_ft30 = np.array(dbp_pred_cerebro_ecg_ft30)

sbp_mae_cerebro_ecg_ft30 = mean_absolute_error(
    sbp_true_cerebro_ecg_ft30,
    sbp_pred_cerebro_ecg_ft30
)

dbp_mae_cerebro_ecg_ft30 = mean_absolute_error(
    dbp_true_cerebro_ecg_ft30,
    dbp_pred_cerebro_ecg_ft30
)

sbp_rmse_cerebro_ecg_ft30 = np.sqrt(mean_squared_error(
    sbp_true_cerebro_ecg_ft30,
    sbp_pred_cerebro_ecg_ft30
))

dbp_rmse_cerebro_ecg_ft30 = np.sqrt(mean_squared_error(
    dbp_true_cerebro_ecg_ft30,
    dbp_pred_cerebro_ecg_ft30
))

sbp_r2_cerebro_ecg_ft30 = r2_score(
    sbp_true_cerebro_ecg_ft30,
    sbp_pred_cerebro_ecg_ft30
)

dbp_r2_cerebro_ecg_ft30 = r2_score(
    dbp_true_cerebro_ecg_ft30,
    dbp_pred_cerebro_ecg_ft30
)

avg_mae_cerebro_ecg_ft30 = (
    sbp_mae_cerebro_ecg_ft30 + dbp_mae_cerebro_ecg_ft30
) / 2

print("Cerebro ECG-Domain Adaptation + Full FT Results")
print("===============================================")
print("Stage A: ECG-only masked reconstruction adaptation")
print("Stage B: Full fine-tuning")
print("LR: 1e-4")
print("Loss: Waveform + SBP/DBP loss")

print("\nSBP")
print(f"MAE  : {sbp_mae_cerebro_ecg_ft30:.3f}")
print(f"RMSE : {sbp_rmse_cerebro_ecg_ft30:.3f}")
print(f"R2   : {sbp_r2_cerebro_ecg_ft30:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_cerebro_ecg_ft30:.3f}")
print(f"RMSE : {dbp_rmse_cerebro_ecg_ft30:.3f}")
print(f"R2   : {dbp_r2_cerebro_ecg_ft30:.4f}")

print("\nTable format:")
print(f"{sbp_mae_cerebro_ecg_ft30:.2f}/{dbp_mae_cerebro_ecg_ft30:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_cerebro_ecg_ft30:.3f}")

Cerebro ECG-Domain Adaptation + Full FT Results
Stage A: ECG-only masked reconstruction adaptation
Stage B: Full fine-tuning
LR: 1e-4
Loss: Waveform + SBP/DBP loss

SBP
MAE  : 6.340
RMSE : 9.189
R2   : 0.6184

DBP
MAE  : 3.103
RMSE : 5.190
R2   : 0.4602

Table format:
6.34/3.10

Average MAE:
4.721
